# End-to-end groundwater imputation and sinkhole-risk workflow

This notebook merges the two supplied reproducible notebooks into a **single in-memory pipeline**:

1. reconstruct missing daily groundwater levels with tuned Extra Trees;
2. pass the reconstructed daily table directly to the monthly sinkhole-risk workflow;
3. engineer monthly predictors and risk classes;
4. fit the validated Random Forest classifier with the reproduced 14-candidate search space;
5. generate manuscript figures, tables, and supplementary diagnostics **only after the modeling workflow has completed**.

The structure follows the two-stage framework of *Sinkhole risk forecasting in the Lithuania–Latvia Karst region using artificial intelligence* (Journal of Hydrology: Regional Studies, 2026).

> **Reproducibility rule:** the supplied executable model logic is preserved. Reporting and notebook organization are rearranged, but the imputation and classification algorithms, random states, feature sets, masking logic, train/test logic, and validated RF grid are not intentionally altered.

> **GitHub-ready version:** run `python scripts/download_data.py --all` from the repository root before executing the notebook. All executable data inputs are then read from relative paths under `data/`.


## Local / Google Colab bootstrap

This setup cell does **not** change either ML workflow. It only locates the repository, installs dependencies when running in Google Colab, and makes the repository data available.


In [ ]:
# Environment/bootstrap only — no modeling logic is changed.
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

REPO_URL = 'https://github.com/VytautasSam/sinkhole_risk_assessment_LT_LV_karst_area'
REPO_FOLDER = REPO_URL.rstrip('/').split('/')[-1].removesuffix('.git')

if IN_COLAB:
    print('Google Colab detected.')

    # Always leave any previous repository directory before Git operations.
    # This prevents getcwd failures if an old clone was deleted.
    os.chdir('/content')

    REPO_ROOT = Path('/content') / REPO_FOLDER

    # Always synchronize the Colab working copy with the current public main
    # branch. This prevents a stale /content clone from silently running an
    # older downloader or notebook support code.
    if not REPO_ROOT.exists():
        print(f'Cloning {REPO_URL} ...')
        subprocess.run(
            ['git', 'clone', REPO_URL, str(REPO_ROOT)],
            check=True,
        )
    else:
        print('Existing Colab repository found; synchronizing with GitHub main...')
        subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'fetch', 'origin', 'main'],
            check=True,
        )
        subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'reset', '--hard', 'origin/main'],
            check=True,
        )

    commit = subprocess.check_output(
        ['git', '-C', str(REPO_ROOT), 'rev-parse', '--short', 'HEAD'],
        text=True,
    ).strip()
    print(f'Using GitHub commit: {commit}')

    os.chdir(REPO_ROOT)

    print('Installing repository requirements...')
    subprocess.run(
        [
            sys.executable,
            '-m',
            'pip',
            'install',
            '-q',
            '-r',
            str(REPO_ROOT / 'requirements.txt'),
        ],
        check=True,
    )

    master_input = REPO_ROOT / 'data' / 'raw' / 'daily_hydroclimate_groundwater.zip'
    sinkhole_input = REPO_ROOT / 'data' / 'raw' / 'sinkholes.csv'

    if not master_input.exists() or not sinkhole_input.exists():
        print()
        print('Core repository data are missing; downloading them now...')
        print('Detailed downloader output follows:')
        print('-' * 70)

        result = subprocess.run(
            [
                sys.executable,
                '-u',
                str(REPO_ROOT / 'scripts' / 'download_data.py'),
                '--core',
            ],
            check=False,
        )

        print('-' * 70)

        if result.returncode != 0:
            raise RuntimeError(
                'Core data download failed. The downloader has printed the '
                'specific failing source and Google Drive/Sheets response above.'
            )

else:
    REPO_ROOT = Path.cwd().resolve()

    if not (REPO_ROOT / 'data').exists() and (REPO_ROOT.parent / 'data').exists():
        REPO_ROOT = REPO_ROOT.parent
        os.chdir(REPO_ROOT)

print(f'Repository root: {REPO_ROOT}')

## Run controls

These switches control **only hyperparameter search**. The scientific workflow, data processing, feature engineering, validation logic, imputation/classification sequence, random states, SHAP calculations, and article outputs remain unchanged.

- `USE_IMPUTATION_GRIDSEARCH = True` — manuscript/reproducible Extra Trees tuning.
- `USE_RISK_GRIDSEARCH = True` — manuscript/reproducible Random Forest tuning with the validated 14-candidate grid.
- Set either switch to `False` only for faster development/testing. The fixed fallback parameters can be edited below.

For reproduction of the reported results, leave **both switches as `True`**.

In [ ]:
from pathlib import Path

# ============================================================
# USER CONTROLS
# ============================================================

USE_IMPUTATION_GRIDSEARCH = True
USE_RISK_GRIDSEARCH = True

# Used only when USE_IMPUTATION_GRIDSEARCH = False.
IMPUTATION_FIXED_PARAMS = {
    'n_estimators': 200,
    'max_depth': None,
    'min_samples_split': 2,
    'min_samples_leaf': 2,
}

# Repository paths. Works locally and after the Colab bootstrap.
if 'REPO_ROOT' not in globals():
    REPO_ROOT = Path.cwd().resolve()
    if not (REPO_ROOT / 'data').exists() and (REPO_ROOT.parent / 'data').exists():
        REPO_ROOT = REPO_ROOT.parent

DATA_RAW = REPO_ROOT / 'data' / 'raw'
DATA_MAP = REPO_ROOT / 'data' / 'map'
FIGURES_DIR = REPO_ROOT / 'figures' / 'generated'
TABLES_DIR = REPO_ROOT / 'tables'

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

## Notebook structure

### Part I — Daily groundwater-level reconstruction
- Load the seven-well daily hydroclimatic/storage dataset.
- Engineer lagged, rolling, seasonal, anomaly, and storage-memory variables.
- Apply the existing groundwater signal cleaning rules.
- Evaluate the `with_seasonal` and `no_seasonal` predictor sets.
- Use contiguous 7–60 day masking, repeated validation, and tuned Extra Trees.
- Reconstruct missing groundwater levels.

### Part II — Monthly sinkhole-risk classification
- Aggregate the reconstructed daily table to monthly resolution.
- Load the sinkhole inventory and calculate monthly occurrence counts.
- Join hydroclimatic/GWL data with sinkhole counts.
- Engineer lagged, rolling, seasonal, groundwater-state and risk variables.
- Evaluate the eight predefined feature sets with the reproduced Random Forest workflow.

### Part III — Article outputs
Figures and tables are grouped at the end in manuscript order. Missing manuscript graphics are kept as clearly marked placeholders for code to be added later.

### Part IV — Supplementary / exploratory diagnostics
VIF tables, feature-importance diagnostics, correlations, and additional scatter plots are kept after the main article outputs.
### Reproducibility controls
The two GridSearch switches at the top can be disabled independently for development runs. Keep both enabled for manuscript reproduction.

### Added article-source figures
Figure 3 is supplied from `1_map(1).ipynb`; Figures 4 and 7, Table 1, and supplementary hydrograph/storage diagnostics are supplied from `4_level(1).ipynb`. These reporting cells are downstream of the main ML pipeline.


Article: [Sinkhole risk forecasting in the Lithuania–Latvia Karst region using artificial intelligence](https://www.sciencedirect.com/science/article/pii/S2214581826002703)


In [ ]:
import io
import os
import json
import logging
import math
import string
import zipfile
from pathlib import Path
from collections import Counter, defaultdict, OrderedDict

import numpy as np
import pandas as pd
import requests
import seaborn as sns
import shap

import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D

from pandas.api.types import is_datetime64_any_dtype, is_period_dtype
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.ensemble import ExtraTreesRegressor, RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import geopandas as gpd
from PIL import Image
from pyproj import Geod
from shapely.geometry import Point
from matplotlib.patches import Patch, Rectangle, FancyArrow

# Part I — Daily groundwater-level reconstruction

The first stage reconstructs gaps in groundwater-level observations using hydroclimatic and remotely sensed storage predictors. The supplied implementation uses two predefined feature sets (`no_seasonal` and `with_seasonal`) and evaluates realistic contiguous missing intervals rather than isolated random points.

The manuscript describes 7–60 day block masking and tuned Extra Trees as the final imputation approach. This notebook preserves the implementation exactly; VIF is retained as a later diagnostic rather than being allowed to change the reproduced predictor sets.

## 1.1 Load and restrict the daily modeling dataset

In [ ]:
MASTER_DATA_ZIP = DATA_RAW / 'daily_hydroclimate_groundwater.zip'
if not MASTER_DATA_ZIP.exists():
    raise FileNotFoundError(
        f'Missing {MASTER_DATA_ZIP}. Run: python scripts/download_data.py --all'
    )

def _first_real_zip_member(z):
    # Preserve the behavior of the original successful notebooks:
    # read the first actual file in the archive, regardless of extension.
    members = [
        info.filename
        for info in z.infolist()
        if not info.is_dir()
        and not info.filename.startswith('__MACOSX/')
        and not info.filename.endswith('.DS_Store')
    ]

    if not members:
        raise RuntimeError(f'No data file found inside {MASTER_DATA_ZIP}')

    return members[0]


with zipfile.ZipFile(MASTER_DATA_ZIP) as z:
    master_member = _first_real_zip_member(z)
    print(f'Reading master dataset from ZIP member: {master_member}')

    with z.open(master_member) as f:
        df = pd.read_csv(f)

wells_of_interest = ['LT_35994', 'LT_35995', 'LT_35996', 'LV_12225', 'LV_22606', 'LV_22610', 'LT_220']
df['date'] = pd.to_datetime(df['date'])
df = df[df['well_no'].isin(wells_of_interest) & (df['date'] >= pd.Timestamp('2003-01-01'))].copy()
df.head()

## 1.2 Daily feature engineering

Lagged variables represent delayed aquifer response, while rolling summaries encode accumulated hydroclimatic memory. Cyclical and seasonal variables represent periodicity without creating a discontinuity between December and January.

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df_orig = df.copy()
df_work = df_orig.copy().sort_values(['well_no', 'date'])

base_features = [
    'precipitation_mm_eobs',
    'temperature_C_eobs',
    'actual_evapotranspiration_mm_gleam',
    'gws_mm_tavg_gldas',
    'gw_level_m_asl',
]
target_col = 'gw_level_m_asl'
lag_days = [30, 60, 90, 180, 270, 360]
roll_days = [60, 90]

all_feature_names = []
for col in base_features:
    for lag in lag_days:
        all_feature_names.append(f'{col}_lag_{lag}d')
    for win in roll_days:
        suffix = 'rollsum' if col in ['precipitation_mm_eobs', 'actual_evapotranspiration_mm_gleam'] else 'rollmean'
        all_feature_names.append(f'{col}_{suffix}_{win}d')

for col in base_features:
    grouped = df_work.groupby('well_no')[col]
    for lag in lag_days:
        df_work[f'{col}_lag_{lag}d'] = grouped.shift(lag)
    for win in roll_days:
        if col == 'precipitation_mm_eobs':
            df_work[f'{col}_rollsum_{win}d'] = grouped.transform(lambda x: x.rolling(win, min_periods=1).sum())
        else:
            df_work[f'{col}_rollmean_{win}d'] = grouped.transform(lambda x: x.rolling(win, min_periods=1).mean())

df_work['dayofyear'] = df_work['date'].dt.dayofyear
df_work['sin_doy'] = np.sin(2 * np.pi * df_work['dayofyear'] / 365.25)
df_work['cos_doy'] = np.cos(2 * np.pi * df_work['dayofyear'] / 365.25)
df_work['month'] = df_work['date'].dt.month
df_work['sin_month'] = np.sin(2 * np.pi * df_work['month'] / 12)
df_work['cos_month'] = np.cos(2 * np.pi * df_work['month'] / 12)
df_work['sin_annual'] = np.sin(2 * np.pi * df_work['date'].dt.dayofyear / 365.25)
df_work['cos_annual'] = np.cos(2 * np.pi * df_work['date'].dt.dayofyear / 365.25)
df_work['seasonal_phase'] = np.arctan2(df_work['sin_doy'], df_work['cos_doy'])

df_work['precip_anomaly'] = (
    df_work['precipitation_mm_eobs_rollsum_90d']
    - df_work.groupby('well_no')['precipitation_mm_eobs'].transform(lambda x: x.rolling(365, min_periods=180).mean())
)
df_work['drought_flag'] = df_work['precip_anomaly'] < -1.5

df_work['gws_monthly_mean'] = df_work.groupby(df_work['date'].dt.month)['gws_mm_tavg_gldas'].transform('mean')
df_work['gws_monthly_anomaly'] = df_work['gws_mm_tavg_gldas'] - df_work['gws_monthly_mean']
df_work['gws_delta_30d'] = df_work['gws_mm_tavg_gldas'] - df_work['gws_mm_tavg_gldas_lag_30d']
df_work['gws_recharge_event'] = df_work['gws_delta_30d'] > df_work['gws_delta_30d'].quantile(0.9)

df_work['below_zero_rolling_30d'] = df_work['temperature_C_eobs'].rolling(30, min_periods=30).apply(
    lambda x: np.mean(x < 0), raw=True
)
df_work['permafrost_flag'] = df_work['below_zero_rolling_30d'] > 0.8

monthly_avg = df_work.groupby(df_work['date'].dt.month).agg({
    'temperature_C_eobs': 'mean',
    'precipitation_mm_eobs': 'mean',
    'gws_mm_tavg_gldas': 'mean',
}).rename(columns={
    'temperature_C_eobs': 'temp_monthly_avg',
    'precipitation_mm_eobs': 'precip_monthly_avg',
    'gws_mm_tavg_gldas': 'gws_monthly_avg',
})

df_work['temp_monthly_avg'] = df_work['month'].map(monthly_avg['temp_monthly_avg'])
df_work['precip_monthly_avg'] = df_work['month'].map(monthly_avg['precip_monthly_avg'])
df_work['gws_monthly_avg'] = df_work['month'].map(monthly_avg['gws_monthly_avg'])

## 1.3 Groundwater signal cleaning

The supplied workflow retains the original groundwater series for reference, removes extreme step changes using the existing threshold rule, applies the LT_35996 post-2022 rule, and fills the specific 2023-12-31 anchor using well-specific December-31 averages.

In [ ]:
df_work['gw_level_m_asl_original'] = df_work['gw_level_m_asl']
df_work = df_work.sort_values(['well_no', 'date'])

diff_forward = df_work.groupby('well_no')['gw_level_m_asl'].diff().abs()
diff_backward = df_work.groupby('well_no')['gw_level_m_asl'].diff(-1).abs()
threshold = df_work['gw_level_m_asl'].diff().abs().quantile(0.9995)
steep_mask = (diff_forward > threshold) | (diff_backward > threshold)

df_work.loc[steep_mask, 'gw_level_m_asl'] = np.nan
df_work.loc[
    (df_work['well_no'] == 'LT_35996') & (df_work['date'] >= pd.Timestamp('2022-06-01')),
    'gw_level_m_asl',
] = np.nan

dec_31_df = df_work[(df_work['date'].dt.month == 12) & (df_work['date'].dt.day == 31)]
avg_gw_levels = dec_31_df.groupby('well_no')['gw_level_m_asl'].mean()
mask = (df_work['date'] == pd.Timestamp('2023-12-31')) & df_work['gw_level_m_asl'].isna()

for idx in df_work[mask].index:
    well = df_work.loc[idx, 'well_no']
    if well in avg_gw_levels:
        df_work.at[idx, 'gw_level_m_asl'] = avg_gw_levels[well]

## 1.4 Imputation feature sets

In [ ]:
feature_sets = {
    'with_seasonal': [
        'precipitation_mm_eobs_rollsum_90d',
        'temperature_C_eobs_lag_30d',
        'temperature_C_eobs_lag_90d',
        'temperature_C_eobs_lag_180d',
        'temperature_C_eobs_lag_270d',
        'temperature_C_eobs_lag_360d',
        'gws_mm_tavg_gldas_lag_90d',
        'gws_mm_tavg_gldas_lag_180d',
        'gws_mm_tavg_gldas_lag_270d',
        'gws_mm_tavg_gldas_lag_360d',
        'gws_mm_tavg_gldas',
        'seasonal_phase',
        'precip_monthly_avg',
    ],
    'no_seasonal': [
        'precipitation_mm_eobs_rollsum_90d',
        'temperature_C_eobs_lag_30d',
        'temperature_C_eobs_lag_90d',
        'temperature_C_eobs_lag_180d',
        'temperature_C_eobs_lag_270d',
        'temperature_C_eobs_lag_360d',
        'gws_mm_tavg_gldas_lag_90d',
        'gws_mm_tavg_gldas_lag_180d',
        'gws_mm_tavg_gldas_lag_270d',
        'gws_mm_tavg_gldas_lag_360d',
        'gws_mm_tavg_gldas',
    ],
}

## 1.5 Extra Trees configuration and block-masking utility

The tuned Extra Trees search space is preserved: 100/200 trees, depth `None`/20, minimum split 2, and minimum leaf size 1/2. The model-selection score is R².

In [ ]:
target_col = 'gw_level_m_asl'
min_samples_required = 5
min_features_ratio = 0.5
eval_fraction = 0.2
cv_fraction = 0.2
cv_repeats = 5
min_mask_length = 7
max_mask_length = 60

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 20],
    'min_samples_split': [2],
    'min_samples_leaf': [1, 2],
}

class FixedExtraTrees:
    """GridSearch-compatible fixed Extra Trees fallback used only when tuning is disabled."""

    def __init__(self, params, random_state=42):
        self.params = dict(params)
        self.random_state = random_state
        self.best_params_ = dict(params)
        self.model_ = None

    def fit(self, X, y):
        self.model_ = ExtraTreesRegressor(random_state=self.random_state, **self.params)
        self.model_.fit(X, y)
        return self

    def predict(self, X):
        return self.model_.predict(X)


if USE_IMPUTATION_GRIDSEARCH:
    imputation_model = GridSearchCV(
        ExtraTreesRegressor(random_state=42),
        param_grid=param_grid,
        cv=3,
        scoring='r2',
        n_jobs=-1,
    )
else:
    imputation_model = FixedExtraTrees(IMPUTATION_FIXED_PARAMS, random_state=42)

models = {'ExtraTreesTuned': imputation_model}

def generate_random_intervals(data, total_target_count, min_len=7, max_len=30, max_attempts=1000):
    data = data.reset_index()
    selected_idx = set()
    attempts = 0
    while len(selected_idx) < total_target_count and attempts < max_attempts:
        interval_len = np.random.randint(min_len, max_len + 1)
        if len(data) <= interval_len:
            break
        start_idx = np.random.randint(0, len(data) - interval_len)
        candidate = data.iloc[start_idx:start_idx + interval_len]['index'].tolist()
        if not selected_idx.intersection(candidate):
            selected_idx.update(candidate)
        attempts += 1
    return list(selected_idx)

## 1.6 Model selection, repeated masked validation, final imputation, and SHAP storage

For each well, the implementation:
- restricts analysis to the observed groundwater period;
- holds out contiguous intervals;
- performs repeated internal block masking;
- tunes Extra Trees with the existing `GridSearchCV`;
- chooses the best feature-set/model combination;
- fits the final model and reconstructs missing groundwater levels;
- stores feature importance, VIF diagnostics, SHAP values, and reproducibility metadata for later reporting.

In [ ]:
imputed_all_models = {}
cv_scores_all = {}
skipped_wells = []
feature_importances_all = {}
vif_all = {}
best_model_per_well = {}
best_model_params = {}
cv_masked_intervals = {}
shap_values_all = {}
shap_importances_all = {}
shap_metadata_all = {}

MAX_SHAP_SAMPLES_PER_WELL = 1000
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

for well_no, group in df_work.groupby('well_no'):
    group = group.copy()
    group['date'] = pd.to_datetime(group['date'])

    if not group[target_col].notnull().any():
        skipped_wells.append(well_no)
        continue

    valid_target = group[target_col].notnull()
    group = group[
        (group['date'] >= group.loc[valid_target, 'date'].min())
        & (group['date'] <= group.loc[valid_target, 'date'].max())
    ]
    data_non_null = group[group[target_col].notnull()].sort_values('date')

    if len(data_non_null) < min_samples_required:
        skipped_wells.append(well_no)
        continue

    eval_idx = generate_random_intervals(
        data_non_null,
        total_target_count=int(eval_fraction * len(data_non_null)),
        min_len=min_mask_length,
        max_len=max_mask_length,
    )
    group.loc[:, '_was_masked'] = False
    group.loc[eval_idx, '_was_masked'] = True

    best_r2, best_key, best_params = -np.inf, None, None

    for feature_set_name, selected_features in feature_sets.items():
        feature_candidates = [f for f in selected_features if f in group.columns]
        if len(feature_candidates) < 2:
            continue

        observed_data = group[~group['_was_masked'] & group[target_col].notnull()].copy()
        if len(observed_data) < min_samples_required:
            continue

        min_required = int(len(feature_candidates) * min_features_ratio)

        for model_name, model_proto in models.items():
            key = (model_name, feature_set_name)
            cv_scores_all.setdefault(key, [])
            imputed_all_models.setdefault(key, [])
            feature_importances_all.setdefault(key, [])
            r2_list, mae_list = [], []

            for rep in range(cv_repeats):
                mask_idx = generate_random_intervals(
                    observed_data,
                    total_target_count=int(cv_fraction * len(observed_data)),
                    min_len=min_mask_length,
                    max_len=max_mask_length,
                )

                if rep == 0:
                    mask_dates = observed_data.loc[mask_idx, 'date'].sort_values()
                    if not mask_dates.empty:
                        gaps = mask_dates.diff().dt.days.fillna(1)
                        segment_ids = (gaps > 1).cumsum()
                        intervals = [(segment.min(), segment.max()) for _, segment in mask_dates.groupby(segment_ids)]
                        cv_masked_intervals[(well_no, model_name, feature_set_name)] = intervals

                train_cv = observed_data.drop(index=mask_idx)
                test_cv = observed_data.loc[mask_idx]
                train_cv = train_cv.dropna(subset=feature_candidates, thresh=min_required)
                test_cv = test_cv.dropna(subset=feature_candidates, thresh=min_required)

                if train_cv.empty or test_cv.empty:
                    continue

                fill_values = train_cv[feature_candidates].median()
                X_train = train_cv[feature_candidates].fillna(fill_values)
                X_test = test_cv[feature_candidates].fillna(fill_values)

                scaler = StandardScaler().fit(X_train)
                X_train_scaled = scaler.transform(X_train)
                X_test_scaled = scaler.transform(X_test)

                model = model_proto.fit(X_train_scaled, train_cv[target_col])
                preds = model.predict(X_test_scaled)
                r2_list.append(r2_score(test_cv[target_col], preds))
                mae_list.append(mean_absolute_error(test_cv[target_col], preds))

            if r2_list:
                r2_mean = np.mean(r2_list)
                mae_mean = np.mean(mae_list)
                cv_scores_all[key].append((well_no, r2_mean, np.std(r2_list), mae_mean, np.std(mae_list)))

                if r2_mean > best_r2:
                    best_r2 = r2_mean
                    best_key = key
                    best_params = model.best_params_

    if best_key is None:
        skipped_wells.append(well_no)
        continue

    model_name, feature_set_name = best_key
    feature_candidates = [f for f in feature_sets[feature_set_name] if f in group.columns]
    min_required = int(len(feature_candidates) * min_features_ratio)

    observed_data = group[~group['_was_masked'] & group[target_col].notnull()].copy()
    fill_values_vif = observed_data[feature_candidates].median()
    scaled_vif_data = observed_data[feature_candidates].fillna(fill_values_vif)
    scaled_vif = StandardScaler().fit_transform(scaled_vif_data)
    vif_scores = [variance_inflation_factor(scaled_vif, i) for i in range(len(feature_candidates))]
    vif_all.setdefault(feature_set_name, []).append(dict(zip(feature_candidates, vif_scores)))

    full_train = group[group[target_col].notnull()]
    full_predict = group[group[target_col].isnull()]
    full_train = full_train.dropna(subset=feature_candidates, thresh=min_required)
    full_predict = full_predict.dropna(subset=feature_candidates, thresh=min_required)

    if full_train.empty:
        skipped_wells.append(well_no)
        continue

    fill_values_full = full_train[feature_candidates].median()
    X_train = full_train[feature_candidates].fillna(fill_values_full)
    y_train = full_train[target_col].values
    X_predict = full_predict[feature_candidates].fillna(fill_values_full) if not full_predict.empty else None

    scaler = StandardScaler().fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_predict_scaled = scaler.transform(X_predict) if X_predict is not None else None

    final_model = ExtraTreesRegressor(random_state=RANDOM_STATE, **best_params)
    final_model.fit(X_train_scaled, y_train)

    if X_predict is not None and len(X_predict) > 0:
        preds = final_model.predict(X_predict_scaled)
        group_copy = group.copy()
        group_copy.loc[full_predict.index, target_col] = preds
        group_copy['feature_set'] = feature_set_name
        imputed_all_models[best_key].append(group_copy)

    if hasattr(final_model, 'feature_importances_'):
        feature_importances_all[best_key].append(dict(zip(feature_candidates, final_model.feature_importances_)))

    best_model_per_well[well_no] = best_key
    best_model_params[well_no] = best_params

    rng = np.random.default_rng(RANDOM_STATE)
    if len(X_train) > MAX_SHAP_SAMPLES_PER_WELL:
        sample_idx = rng.choice(len(X_train), size=MAX_SHAP_SAMPLES_PER_WELL, replace=False)
        X_train_sampled = X_train.iloc[sample_idx].reset_index(drop=True)
        X_train_scaled_sampled = X_train_scaled[sample_idx, :]
        y_train_sampled = y_train[sample_idx]
    else:
        X_train_sampled = X_train.reset_index(drop=True)
        X_train_scaled_sampled = X_train_scaled
        y_train_sampled = y_train

    explainer = shap.TreeExplainer(final_model)
    shap_vals = explainer.shap_values(X_train_scaled_sampled)
    base_val = explainer.expected_value

    shap_values_all.setdefault(best_key, []).append({
        'well_no': well_no,
        'feature_names': feature_candidates,
        'shap_values': shap_vals,
        'base_value': base_val,
        'X_features': X_train_sampled[feature_candidates],
        'y': y_train_sampled,
    })
    shap_importances_all.setdefault(best_key, []).append(
        dict(zip(feature_candidates, np.abs(shap_vals).mean(axis=0)))
    )
    shap_metadata_all.setdefault(best_key, []).append({
        'well_no': well_no,
        'n_samples': len(X_train_sampled),
        'feature_set_name': feature_set_name,
        'model_name': model_name,
        'fill_values': fill_values_full.to_dict(),
        'scaler_type': 'StandardScaler',
    })

rows = []
for well_no, best_key in best_model_per_well.items():
    model_name, feature_set_name = best_key
    metrics = next((t for t in cv_scores_all.get(best_key, []) if t[0] == well_no), None)
    if metrics is None:
        r2_mean = r2_std = mae_mean = mae_std = float('nan')
    else:
        _, r2_mean, r2_std, mae_mean, mae_std = metrics

    rows.append({
        'well_no': well_no,
        'CV_R2 mean': r2_mean,
        'CV_R2 std': r2_std,
        'CV_MAE mean': mae_mean,
        'CV_MAE std': mae_std,
        'feature_set_name': feature_set_name,
        'best_params': best_model_params.get(well_no),
    })

summary = pd.DataFrame(rows).sort_values('CV_R2 mean', ascending=False)
avg = {
    'well_no': 'AVERAGE',
    'CV_R2 mean': summary['CV_R2 mean'].mean(),
    'CV_R2 std': summary['CV_R2 std'].mean(),
    'CV_MAE mean': summary['CV_MAE mean'].mean(),
    'CV_MAE std': summary['CV_MAE std'].mean(),
    'feature_set_name': '',
    'best_params': '',
}
summary = pd.concat([summary, pd.DataFrame([avg])], ignore_index=True)
# Table 2 is displayed in the article-output section below.

## 1.7 In-memory handoff to the risk model

The original two-notebook workflow wrote the reconstructed table to Parquet and then read it in the risk notebook. That serialization step is removed here. The **same merged daily table is passed directly in memory** to the second stage.

In [ ]:
# Build the continuous daily groundwater-level table in memory.
# This is the same merge that previously produced the intermediate Parquet file,
# but the result is passed directly to the sinkhole-risk workflow.

df_imputed_vals = pd.concat(
    [frame for frames in imputed_all_models.values() for frame in frames],
    ignore_index=True,
)

selected_cols = df_imputed_vals[['well_no', 'date', 'gw_level_m_asl']].rename(
    columns={'gw_level_m_asl': 'gw_level_m_asl_imputed'}
)

df_merged = df_work.merge(selected_cols, on=['well_no', 'date'], how='left')

# The risk notebook previously read this table from Parquet into `df`.
# Keep the same variable name, but stay entirely in memory.
df = df_merged.copy()

print(f'In-memory handoff complete: {df.shape[0]:,} rows × {df.shape[1]:,} columns')
print('Wells:', sorted(df['well_no'].dropna().unique()))

# Part II — Monthly sinkhole-risk classification

The second stage converts the reconstructed daily hydroclimatic/GWL table to monthly resolution and combines it with monthly sinkhole counts. The classifier distinguishes **Low risk** and **High risk** months using the same threshold and feature-engineering logic as the supplied reproducible risk notebook.

## 2.1 Aggregate daily predictors and reconstructed groundwater level to monthly resolution

In [ ]:
df_level = df[[
    'date',
    'well_no',
    'precipitation_mm_eobs',
    'temperature_C_eobs',
    'actual_evapotranspiration_mm_gleam',
    'potential_evapotranspiration_mm_gleam',
    'tws_mm_tavg_gldas',
    'gws_mm_tavg_gldas',
    'gw_level_m_asl_imputed'
]].copy()
# Ensure 'date' is datetime
df_level['date'] = pd.to_datetime(df_level['date'])

# Extract year-month string (e.g., "2025-08")
df_level['year_month'] = df_level['date'].dt.to_period('M').astype(str)

# Group by well_no and year_month
df_level_monthly = df_level.groupby(['well_no', 'year_month']).agg({
    'temperature_C_eobs': 'mean',
    'actual_evapotranspiration_mm_gleam': 'mean',
    'potential_evapotranspiration_mm_gleam': 'mean',
    'tws_mm_tavg_gldas': 'mean',
    'gws_mm_tavg_gldas': 'mean',
    'gw_level_m_asl_imputed': 'mean',
    'precipitation_mm_eobs': 'sum'
}).reset_index()
df_level_monthly.head()

## 2.2 Load the sinkhole inventory

In [ ]:
SINKHOLE_CSV = DATA_RAW / 'sinkholes.csv'
if not SINKHOLE_CSV.exists():
    raise FileNotFoundError(
        f'Missing {SINKHOLE_CSV}. Run: python scripts/download_data.py --all'
    )

df_sink = pd.read_csv(SINKHOLE_CSV)
df_sink.head()

## 2.3 Build the monthly modeling table

Sinkhole formation dates are aggregated to monthly counts, the temporal key is normalized to `Period[M]`, and the occurrence series is left-joined to every monitoring well. Months without recorded sinkholes are retained as zero-count months.

In [ ]:
def to_month_period(s):
    """Coerce a Series/Index to monthly Period[M] without deprecated checks."""
    dtype = getattr(s, "dtype", None)

    if isinstance(dtype, pd.PeriodDtype):
        # Already Period -> just enforce monthly freq
        return s.astype("period[M]")
    elif is_datetime64_any_dtype(s):
        # Datetime -> monthly Period
        return s.dt.to_period("M") if hasattr(s, "dt") else s.to_period("M")
    else:
        # Strings/objects -> parse to datetime, then to Period
        s_dt = pd.to_datetime(s, errors="coerce")
        return s_dt.dt.to_period("M") if hasattr(s_dt, "dt") else s_dt.to_period("M")


# --- Build monthly counts with a true monthly Period key ---
df_sink = df_sink[df_sink['date_event'].notna()].copy()
df_sink['date_event'] = pd.to_datetime(df_sink['date_event'])
df_sink['year_month'] = df_sink['date_event'].dt.to_period('M')

df_sink_count_monthly = (
    df_sink.groupby('year_month')
           .size()
           .rename('sink_count')
           .reset_index()
)

# --- Make BOTH sides monthly Period[M] (no to_datetime on Period!) ---
df_level_monthly['year_month']       = to_month_period(df_level_monthly['year_month'])
df_sink_count_monthly['year_month']  = to_month_period(df_sink_count_monthly['year_month'])

# --- Merge (fix the name if you had a typo earlier) ---
df_work = pd.merge(
    df_level_monthly,
    df_sink_count_monthly,  # not df_sink_plot_count_monthly
    on='year_month',
    how='left'
)

# --- Post-merge columns ---
df_work['sink_count']     = df_work['sink_count'].fillna(0).astype(int)

# Period accessors now work
df_work['year']           = df_work['year_month'].dt.year
df_work['month']          = df_work['year_month'].dt.month
df_work['month_name']     = df_work['year_month'].dt.to_timestamp().dt.month_name()

# Keep a string label for display, but don’t overwrite the Period key
df_work['year_month_str'] = df_work['year_month'].astype(str)  # 'YYYY-MM'

# If you also need a datetime for plotting (month start):
df_work['year_month_ts']  = df_work['year_month'].dt.to_timestamp()

df_work.head()

## 2.4 Risk threshold and monthly predictor engineering

The reproduced classifier uses a threshold value of **4** in the supplied implementation. With the existing `pd.cut(..., right=True)` logic, months above this threshold are mapped to the High-risk class.

Predictors include groundwater state, groundwater seasonality, climate, evapotranspiration ratios, storage variables, lagged/rolling terms, and freezing indicators. Shifted rolling features are retained exactly as implemented to avoid using the current month in antecedent summaries.

In [ ]:
threshold = 4

# === Lag and Rolling Feature Configuration ===
lag_features = [
    'gw_level_m_asl_imputed',
    'temperature_C_eobs',
    'precipitation_mm_eobs',
    'actual_evapotranspiration_mm_gleam',
    'potential_evapotranspiration_mm_gleam',
    'tws_mm_tavg_gldas',
    'gws_mm_tavg_gldas',
]

n_lags = 3           # Number of lag months
rolling_window = 3   # Rolling window size

rolling_mean_features = ['gw_level_m_asl_imputed', 'temperature_C_eobs', 'tws_mm_tavg_gldas', 'gws_mm_tavg_gldas',]
rolling_sum_features = ['precipitation_mm_eobs', 'actual_evapotranspiration_mm_gleam', 'potential_evapotranspiration_mm_gleam']

# === Generate Lag and Rolling Features ===
for feature in lag_features:
    # Lag features (1 to n_lags months back)
    for lag in range(1, n_lags + 1):
        df_work[f'{feature}_lag_{lag}'] = df_work.groupby('well_no')[feature].shift(lag)

    # Rolling Mean or Sum Features (with 1-month shift to avoid data leakage)
    if feature in rolling_mean_features:
        df_work[f'{feature}_roll_mean_{rolling_window}'] = (
            df_work.groupby('well_no')[feature]
            .transform(lambda x: x.shift(1).rolling(window=rolling_window, min_periods=1).mean())
        )
    elif feature in rolling_sum_features:
        df_work[f'{feature}_roll_sum_{rolling_window}'] = (
            df_work.groupby('well_no')[feature]
            .transform(lambda x: x.shift(1).rolling(window=rolling_window, min_periods=1).sum())
        )

# === Encode Seasonality with Sine/Cosine Transformation ===
df_work['month_sin'] = np.sin(2 * np.pi * df_work['month'] / 12)
df_work['month_cos'] = np.cos(2 * np.pi * df_work['month'] / 12)

# Seasonal phase angle in radians
df_work['seasonal_phase'] = np.arctan2(df_work['month_sin'], df_work['month_cos'])

# Seasonal modulation of evapotranspiration (optional)
df_work['seasonal_evapo_effect'] = df_work['actual_evapotranspiration_mm_gleam'] * df_work['seasonal_phase']

# === Groundwater Seasonality-Adjusted Features ===

# Groundwater level above/below well average
df_work['gw_level_above_avg'] = df_work.groupby('well_no')['gw_level_m_asl_imputed'].transform(
    lambda x: x > x.mean()
).astype(int)

# Whether GW level is near recent 6-month peak (centered)
df_work['gw_level_near_peak'] = df_work.groupby('well_no')['gw_level_m_asl_imputed'].transform(
    lambda x: x.rolling(window=6, center=True).max() == x
).astype(int)

# One-month delta (rate of change)
df_work['gw_level_delta_1'] = df_work['gw_level_m_asl_imputed'] - df_work['gw_level_m_asl_imputed_lag_1']

# Rolling 3-month trend (slope-like)
df_work['gw_level_trend_3'] = df_work.groupby('well_no')['gw_level_m_asl_imputed'].transform(
    lambda x: x.diff().rolling(3).mean()
)

# Seasonal modulation: Multiply GW level by seasonal components to capture cyclical effects
df_work['gw_level_seasonal_sin'] = df_work['gw_level_m_asl_imputed'] * df_work['month_sin']
df_work['gw_level_seasonal_cos'] = df_work['gw_level_m_asl_imputed'] * df_work['month_cos']
df_work['gw_level_seasonal_phase'] = df_work['gw_level_m_asl_imputed'] * df_work['seasonal_phase']

# === Evapotranspiration Ratio ===
df_work['evapo_ratio'] = df_work['actual_evapotranspiration_mm_gleam'] / (
    df_work['potential_evapotranspiration_mm_gleam'] + 1e-5
)

# === Freezing Conditions Analysis ===
df_work['is_freezing'] = (df_work['temperature_C_eobs'] <= 0).astype(int)

def compute_consecutive_freezing(series):
    max_consec = []
    count = 0
    for val in series:
        if val == 1:
            count += 1
        else:
            count = 0
        max_consec.append(count)
    return max_consec

df_work['consec_freezing'] = df_work.groupby('well_no')['is_freezing'].transform(compute_consecutive_freezing)

# === Groundwater Spike Features ===

# Daily diff
df_work['gw_level_diff'] = df_work['gw_level_m_asl_imputed'].diff()

# 3-month cumulative increase
df_work['gw_level_rolling_increase'] = df_work['gw_level_diff'].rolling(window=3).sum()

# Flag if sudden spike > 0.5 meters
df_work['gw_level_spike'] = (df_work['gw_level_diff'] > 0.5).astype(int)

# Count of such spikes over past 6 months
df_work['gw_spike_count_6m'] = df_work['gw_level_spike'].rolling(window=6).sum()

# === Sink Count to Risk Mapping ===

# Categorize based on sink count
bins = [0, threshold, float('inf')]
categories = [0, 1]

df_work['sink_category'] = pd.cut(
    df_work['sink_count'],
    bins=bins,
    labels=categories,
    right=True,
    include_lowest=True
)

# Map to human-readable risk labels
risk_map = {
    0: "Low risk",
    1: "High risk",
}
df_work['risk_level'] = df_work['sink_category'].astype(int).map(risk_map)

## 2.5 Predefined sinkhole-risk feature sets

In [ ]:
feature_sets = {
    "Groundwater Features": [
        'gw_level_m_asl_imputed',
        # 'gw_level_m_asl_imputed_roll_mean_3',
        'gw_level_above_avg',
        # 'gw_level_rolling_increase',
        'gw_level_spike',
    ],
    "Climatic Features": [
        # 'temperature_C_eobs',
        'precipitation_mm_eobs',
        'actual_evapotranspiration_mm_gleam',
        'evapo_ratio',
        # 'temperature_C_eobs_lag_1',
        'precipitation_mm_eobs_lag_1',
        'actual_evapotranspiration_mm_gleam_lag_1',
        # 'temperature_C_eobs_lag_3',
        'precipitation_mm_eobs_lag_3',
        # 'actual_evapotranspiration_mm_gleam_lag_3',
    ],
    # Groundwater-Seasonal Combined Features
    "GS Combined Features": [
        'gw_level_m_asl_imputed',
        'gw_level_seasonal_phase',
        'gw_level_spike',
        'month_sin',
    ],
    # Climatic-Seasonal Combined Features
    "CS Combined Features": [
        'evapo_ratio',
        'temperature_C_eobs_roll_mean_3',
        'precipitation_mm_eobs_roll_sum_3',
        'seasonal_evapo_effect',
    ],
    "TWS Features": [
        'tws_mm_tavg_gldas',
        'tws_mm_tavg_gldas_roll_mean_3',
    ],
    "GWS Features": [
        'gws_mm_tavg_gldas',
        'gws_mm_tavg_gldas_roll_mean_3',
    ],
    "GWS-Seasonal-Features": [
        'gws_mm_tavg_gldas',
        'gws_mm_tavg_gldas_roll_mean_3',
        'month_sin',
    ],
    # Climatic-Groundwater-Seasonal Combined Features
    "CGS Combined Features": [
        'evapo_ratio',
        'temperature_C_eobs_roll_mean_3',
        'precipitation_mm_eobs_roll_sum_3',
        'actual_evapotranspiration_mm_gleam',
        # 'seasonal_evapo_effect',
        'gw_level_m_asl_imputed',
        'gw_level_seasonal_phase',
        # 'month_sin',
        # 'gws_mm_tavg_gldas',
        # 'gws_mm_tavg_gldas_roll_mean_3',
    ],
}

## 2.6 Reproduced Random Forest classification

This is the **validated 14-candidate version** that reproduced the original 36-candidate results in the previous checks. The original candidate order is preserved. The modeling block keeps the supplied time split, inner `TimeSeriesSplit`, encoding, class-imbalance handling, Random Forest fitting, metrics, and SHAP calculations.

In [ ]:
# -------------------- High-level toggles --------------------
USE_GRIDSEARCH = USE_RISK_GRIDSEARCH  # controlled at the top of the notebook
N_JOBS = -1               # parallel jobs for GridSearchCV
VERBOSE = 1               # verbosity for GridSearchCV output

# Default RF hyperparameters (used when USE_GRIDSEARCH=False)
DEFAULT_RF_PARAMS = dict(
    n_estimators=200,
    max_depth=30,
    min_samples_leaf=3,
    class_weight='balanced',
    random_state=43
)

# Search space (used when USE_GRIDSEARCH=True)
# Exact union of the 14 configurations selected in the successful
# original 36-candidate run across all 56 well × feature-set searches.
# Relative candidate order is preserved from the original grid.
PARAM_GRID = [
    {
        'model__n_estimators': [200],
        'model__max_depth': [20],
        'model__min_samples_leaf': [2],
        'model__class_weight': ['balanced_subsample'],
    },
    {
        'model__n_estimators': [400],
        'model__max_depth': [20],
        'model__min_samples_leaf': [2],
        'model__class_weight': ['balanced_subsample'],
    },
    {
        'model__n_estimators': [200],
        'model__max_depth': [20],
        'model__min_samples_leaf': [3],
        'model__class_weight': ['balanced_subsample'],
    },
    {
        'model__n_estimators': [400],
        'model__max_depth': [20],
        'model__min_samples_leaf': [3],
        'model__class_weight': ['balanced_subsample'],
    },
    {
        'model__n_estimators': [200],
        'model__max_depth': [20],
        'model__min_samples_leaf': [5],
        'model__class_weight': ['balanced_subsample'],
    },
    {
        'model__n_estimators': [400],
        'model__max_depth': [20],
        'model__min_samples_leaf': [5],
        'model__class_weight': ['balanced_subsample'],
    },
    {
        'model__n_estimators': [200],
        'model__max_depth': [20],
        'model__min_samples_leaf': [3],
        'model__class_weight': [{0: 1, 1: 4}],
    },
    {
        'model__n_estimators': [400],
        'model__max_depth': [20],
        'model__min_samples_leaf': [3],
        'model__class_weight': [{0: 1, 1: 4}],
    },
    {
        'model__n_estimators': [200],
        'model__max_depth': [20],
        'model__min_samples_leaf': [5],
        'model__class_weight': [{0: 1, 1: 4}],
    },
    {
        'model__n_estimators': [400],
        'model__max_depth': [20],
        'model__min_samples_leaf': [5],
        'model__class_weight': [{0: 1, 1: 4}],
    },
    {
        'model__n_estimators': [200],
        'model__max_depth': [20],
        'model__min_samples_leaf': [2],
        'model__class_weight': [{0: 1, 1: 6}],
    },
    {
        'model__n_estimators': [400],
        'model__max_depth': [20],
        'model__min_samples_leaf': [2],
        'model__class_weight': [{0: 1, 1: 6}],
    },
    {
        'model__n_estimators': [200],
        'model__max_depth': [20],
        'model__min_samples_leaf': [5],
        'model__class_weight': [{0: 1, 1: 6}],
    },
    {
        'model__n_estimators': [400],
        'model__max_depth': [20],
        'model__min_samples_leaf': [5],
        'model__class_weight': [{0: 1, 1: 6}],
    },
]

# Cross-validation/config
random_state = 43
n_splits = 5  # desired inner CV splits (will be clamped if data is small)
logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger(__name__)


# -------------------- Simple encoder --------------------
class SimpleEncoder(BaseEstimator, TransformerMixin):
    """Label-encodes object/category columns and leaves numeric columns as-is.

    Robust to unseen categories at transform time by mapping them to '__UNK__'.
    """
    def __init__(self):
        self.encoders = {}
        self.categorical_cols_ = None

    def fit(self, X, y=None):
        X = X.copy()
        self.categorical_cols_ = [
            c for c in X.columns
            if X[c].dtype == 'object' or str(X[c].dtype).startswith('category')
        ]
        for col in self.categorical_cols_:
            le = LabelEncoder()
            vals = X[col].astype(str).fillna("")
            uniques = pd.unique(vals)
            # include an explicit unknown bucket for robustness
            classes = np.concatenate([uniques, np.array(['__UNK__'])])
            le.fit(classes)
            self.encoders[col] = le
        return self

    def transform(self, X):
        X = X.copy()
        for col, le in self.encoders.items():
            vals = X[col].astype(str).fillna("")
            mask_unseen = ~np.isin(vals, le.classes_)
            if np.any(mask_unseen):
                vals.loc[mask_unseen] = '__UNK__'
            X[col] = le.transform(vals)
        return X


# -------------------- Utilities --------------------

def ensure_datetime(series):
    """
    Coerce to datetime64[ns].
    - If dtype is Period (e.g., period[M]), convert to the START of the period.
    """
    s = series if isinstance(series, pd.Series) else pd.Series(series)
    dtype = s.dtype

    # Period dtype -> convert to timestamp at the start of the period
    try:
        if isinstance(dtype, pd.PeriodDtype):
            return s.dt.to_timestamp(how="start")
    except Exception:
        pass

    # Datetime-like already? normalize via to_datetime
    if is_datetime64_any_dtype(dtype):
        return pd.to_datetime(s, errors="coerce")

    # Fallback: parse strings/ints/etc.
    return pd.to_datetime(s, errors="coerce")


def safe_roc_auc(y_true, y_proba):
    """ROC AUC only defined if both classes present; otherwise returns NaN."""
    try:
        if y_proba is not None and len(y_proba) == len(y_true):
            return roc_auc_score(y_true, y_proba)
        return np.nan
    except Exception:
        return np.nan


def compute_adaptive_k(y):
    """Choose a safe SMOTE k_neighbors from y."""
    class_counts = Counter(y)
    n_min = min(class_counts.values())
    # SMOTE requires at least k+1 minority samples; cap at 5
    k = max(1, min(5, n_min - 1))
    # If not enough minority samples to SMOTE (n_min < 2), return None to skip SMOTE
    return k if n_min >= 2 else None


def make_timeseries_inner_cv(n_samples, desired_splits=5):
    """
    Create a TimeSeriesSplit with a valid number of splits for n_samples.
    TimeSeriesSplit requires n_splits >= 2 and < n_samples.
    """
    if n_samples <= 3:
        return None
    n_splits_valid = min(desired_splits, max(2, n_samples - 1))
    if n_splits_valid >= n_samples:
        return None
    return TimeSeriesSplit(n_splits=n_splits_valid)


def _compute_tree_shap(model: RandomForestClassifier, X_enc: pd.DataFrame):
    """Compute SHAP values for a fitted tree model on encoded data.

    Ensures a 2D shape (n_samples, n_features) for SHAP values, picking the
    positive class when multi-class is present, or reducing extra dims.

    Returns (shap_values, base_values):
        - shap_values: np.ndarray [n_samples, n_features] for the positive class (1)
        - base_values: np.ndarray [n_samples] expected value replicated per sample
    """
    if X_enc is None or X_enc.shape[0] == 0:
        return None, None
    expl = shap.TreeExplainer(model)
    sv_raw = expl.shap_values(X_enc)

    # Normalize to a 2D array (n_samples, n_features)
    if isinstance(sv_raw, list):
        # pick positive class if available
        sv = np.array(sv_raw[1 if len(sv_raw) > 1 else 0])
        base = expl.expected_value[1 if (hasattr(expl, 'expected_value') and isinstance(expl.expected_value, (list, tuple)) and len(expl.expected_value) > 1) else 0]
    else:
        sv = np.array(sv_raw)
        base = expl.expected_value if np.ndim(expl.expected_value) == 0 else np.array(expl.expected_value).ravel()[0]

    # If still 3D, try to squeeze the class dimension
    if sv.ndim == 3:
        # common cases: (n_samples, n_features, n_classes) or (n_classes, n_samples, n_features)
        if sv.shape[0] == X_enc.shape[0]:
            # (n_samples, n_features, n_classes)
            sv = sv[:, :, 1 if sv.shape[2] > 1 else 0]
        elif sv.shape[1] == X_enc.shape[0]:
            # (n_classes, n_samples, n_features)
            sv = sv[1 if sv.shape[0] > 1 else 0, :, :]
        else:
            # fallback: average over the smallest axis (assumed class axis)
            class_axis = int(np.argmin([abs(d - X_enc.shape[0]) for d in sv.shape]))
            sv = np.take(sv, indices=0, axis=class_axis)

    # If 1D (single feature), make it 2D
    if sv.ndim == 1:
        sv = sv.reshape(-1, 1)

    base_vals = np.full(X_enc.shape[0], fill_value=float(base))
    return sv, base_vals


# -------------------- Core classification (per well; time-aware) --------------------

def run_classification(
    df: pd.DataFrame,
    label_col: str,
    features: list[str],
    time_col: str = 'year_month',
    use_gridsearch: bool = USE_GRIDSEARCH,
    param_grid: dict | None = PARAM_GRID,
    default_rf_params: dict | None = DEFAULT_RF_PARAMS
):
    """
    Time-aware training/evaluation:
      - TEST = rows in the two newest calendar years per well (by time_col).
      - TRAIN = all earlier rows.
      - Inner CV = TimeSeriesSplit on TRAIN only (for GridSearch if enabled).
      - Final pipeline: [encoder -> SMOTETomek(if feasible) -> RF] fit on TRAIN; evaluate on TEST.

    Returns:
        y_true_test, y_pred_test, y_proba_test, selected_features,
        X_test_enc (DataFrame), shap_values (np.ndarray), base_values (np.ndarray), best_params_overall
    """
    y_true_all, y_pred_all, y_proba_all = [], [], []
    selected_features = None
    X_test_enc = None
    shap_values = None
    base_values = None
    best_params_overall = None

    # Filter rows having all required cols and valid time
    needed = list(set(features + [label_col, time_col]))
    df_filtered = df.dropna(subset=[c for c in needed if c in df.columns]).copy()
    if df_filtered.empty:
        return y_true_all, y_pred_all, y_proba_all, selected_features, X_test_enc, shap_values, base_values, best_params_overall

    # Coerce time; drop NaT
    df_filtered[time_col] = ensure_datetime(df_filtered[time_col])
    df_filtered = df_filtered.dropna(subset=[time_col])
    if df_filtered.empty:
        return y_true_all, y_pred_all, y_proba_all, selected_features, X_test_enc, shap_values, base_values, best_params_overall

    # Sort by time
    df_filtered = df_filtered.sort_values(time_col).reset_index(drop=True)

    # Binary labels
    if df_filtered[label_col].nunique() < 2:
        return y_true_all, y_pred_all, y_proba_all, selected_features, X_test_enc, shap_values, base_values, best_params_overall

    # Train/Test by **four newest years**
    years = df_filtered[time_col].dt.year.dropna().astype(int)
    uniq_years_sorted = np.sort(years.unique())
    if len(uniq_years_sorted) == 0:
        return y_true_all, y_pred_all, y_proba_all, selected_features, X_test_enc, shap_values, base_values, best_params_overall

    test_years = set(uniq_years_sorted[-4:])  # <-- Four newest years
    is_test = years.isin(test_years)
    df_train = df_filtered.loc[~is_test].copy()
    df_test  = df_filtered.loc[is_test].copy()

    # Need at least 1 sample in test and at least 2 classes in train to proceed
    if df_test.empty or df_train.empty or df_train[label_col].nunique() < 2:
        return y_true_all, y_pred_all, y_proba_all, selected_features, X_test_enc, shap_values, base_values, best_params_overall

    X_train = df_train[features].reset_index(drop=True)
    y_train = df_train[label_col].reset_index(drop=True)
    X_test  = df_test[features].reset_index(drop=True)
    y_test  = df_test[label_col].reset_index(drop=True)

    # -------------------- Feature selection (skipped) --------------------
    # With <=5 features per set, we keep all features as-is.
    selected_features = list(X_train.columns)

    # -------------------- Hyperparameters (inner CV on TRAIN) --------------------
    if use_gridsearch:
        inner_cv = make_timeseries_inner_cv(n_samples=len(X_train), desired_splits=n_splits)
        if inner_cv is None:
            logger.info("GridSearch requested but training set is too small for TimeSeriesSplit; using defaults.")
            rf_params = dict(default_rf_params)
        else:
            gs_pipe = Pipeline([
                ('encoder', SimpleEncoder()),
                ('model', RandomForestClassifier(random_state=random_state))
            ])
            grid = GridSearchCV(
                gs_pipe, param_grid=param_grid, cv=inner_cv,
                scoring='roc_auc', n_jobs=N_JOBS, verbose=VERBOSE
            )
            grid.fit(X_train[selected_features], y_train)
            best_params = grid.best_params_
            rf_params = {k.split("model__", 1)[1]: v for k, v in best_params.items()}
            rf_params['random_state'] = random_state
            best_params_overall = dict(rf_params)
    else:
        rf_params = dict(default_rf_params)
        best_params_overall = dict(rf_params)

    # -------------------- Final pipeline with SMOTETomek (fit on TRAIN only) --------------------
    k = compute_adaptive_k(y_train)
    if k is not None:
        sampler = SMOTETomek(smote=SMOTE(k_neighbors=k, random_state=random_state),
                             random_state=random_state)
        final_pipe = ImbPipeline([
            ('encoder', SimpleEncoder()),
            ('sampler', sampler),
            ('model', RandomForestClassifier(**rf_params))
        ])
    else:
        final_pipe = ImbPipeline([
            ('encoder', SimpleEncoder()),
            ('model', RandomForestClassifier(**rf_params))
        ])

    # Fit on TRAIN (selected features), predict TEST
    final_pipe.fit(X_train[selected_features], y_train)
    y_pred = final_pipe.predict(X_test[selected_features])
    y_proba = None
    if hasattr(final_pipe.named_steps['model'], "predict_proba"):
        y_proba = final_pipe.predict_proba(X_test[selected_features])[:, 1]

    # Collect outputs
    y_true_all.extend(y_test)
    y_pred_all.extend(y_pred)
    if y_proba is not None:
        y_proba_all.extend(y_proba)

    # Prepare encoded TEST matrix for SHAP
    X_test_enc = final_pipe.named_steps['encoder'].transform(X_test[selected_features].copy())

    # Compute SHAP on TEST (positive class)
    shap_values, base_values = _compute_tree_shap(final_pipe.named_steps['model'], X_test_enc)

    return (
        y_true_all, y_pred_all, y_proba_all,
        selected_features, X_test_enc, shap_values, base_values,
        best_params_overall
    )


# -------------------- Evaluation across feature sets --------------------

def evaluate_feature_sets(df_work: pd.DataFrame, feature_sets: dict[str, list[str]], time_col: str = 'year_month'):
    """
    df_work: DataFrame with at least columns:
        - 'well_no' (grouping key)
        - 'year_month' (time column)
        - 'gw_level_m_asl_imputed' (for temporal features; optional)
        - 'risk_level' with values 'Low risk' / 'High risk'
        - plus all features referenced in `feature_sets`

    feature_sets: dict {name: list_of_feature_columns}

    Returns a dict with:
        - general_summary (DataFrame)
        - per_well_accuracy (DataFrame)
        - per_well_tables (dict[str, DataFrame])
        - confusion_matrices (list[(feature_set, np.array)])
        - best_config_per_well (DataFrame)
        - shap_aggregated (dict[str, dict])  # aggregated SHAP per feature set for plotting
    """
    per_well_tables = {}
    general_summary = []
    confusion_matrices = []
    per_well_accuracy_combined = []

    # NEW: track best configuration per well across all feature sets
    best_config_by_well = {}

    # SHAP collectors
    shap_collector = defaultdict(lambda: {"X_list": [], "SV_list": [], "BV_list": [], "feature_names": None})

    def _primary_score(roc_auc, acc):
        if roc_auc is not None and not np.isnan(roc_auc):
            return float(roc_auc), "ROC AUC"
        return float(acc) if acc is not None else np.nan, "Accuracy"

    def _is_better(candidate, current):
        cand_score, _ = _primary_score(candidate['roc_auc'], candidate['accuracy'])
        curr_score, _ = _primary_score(current['roc_auc'], current['accuracy'])
        if cand_score > curr_score + 1e-12:
            return True
        if abs(cand_score - curr_score) <= 1e-12:
            if candidate['accuracy'] > current['accuracy'] + 1e-12:
                return True
            if abs(candidate['accuracy'] - current['accuracy']) <= 1e-12:
                return candidate.get('n_samples', 0) > current.get('n_samples', 0)
        return False

    # Ensure risk label is present
    if 'risk_level' not in df_work.columns:
        raise ValueError("df_work must contain a 'risk_level' column with 'Low risk'/'High risk' values.")

    # Work on a copy with binary label prepared once
    df_work = df_work.copy()
    df_work['year_month'] = ensure_datetime(df_work[time_col])
    df_work = df_work.dropna(subset=['year_month'])
    df_work = df_work[df_work['risk_level'].isin(['Low risk', 'High risk'])].copy()
    if df_work.empty:
        logger.warning("No rows with valid 'year_month' and risk labels.")
        return {
            "general_summary": pd.DataFrame(),
            "per_well_accuracy": pd.DataFrame(),
            "per_well_tables": {},
            "confusion_matrices": [],
            "best_config_per_well": pd.DataFrame(),
            "shap_aggregated": {}
        }
    df_work['binary_label'] = df_work['risk_level'].map({'Low risk': 0, 'High risk': 1})

    os.makedirs("shap_values", exist_ok=True)

    for name, features in feature_sets.items():
        logger.info(f"\n>>> Evaluating feature set: {name}")
        all_true, all_preds, all_probas = [], [], []
        well_results = []

        # Validate features exist
        missing = [f for f in features if f not in df_work.columns]
        if missing:
            logger.warning(f"Skipping feature set '{name}': missing columns: {missing}")
            continue

        # Prepare per-feature-set SHAP directory
        per_set_dir = os.path.join("shap_values", name)
        os.makedirs(per_set_dir, exist_ok=True)

        for well_id, df_well in df_work.groupby("well_no"):
            df_well = df_well.sort_values('year_month').copy()

            # --- Optional temporal/derived features from groundwater level ---
            if 'gw_level_m_asl_imputed' in df_well.columns:
                df_well['gw_level_diff'] = df_well['gw_level_m_asl_imputed'].diff()
                df_well['gw_level_rolling_increase'] = df_well['gw_level_diff'].rolling(window=3, min_periods=1).sum()
                df_well['gw_level_spike'] = (df_well['gw_level_diff'] > 0.5).astype(int)
                df_well['gw_spike_count_6m'] = df_well['gw_level_spike'].rolling(window=6, min_periods=1).sum()
            else:
                for col in ['gw_level_diff', 'gw_level_rolling_increase', 'gw_level_spike', 'gw_spike_count_6m']:
                    if col not in df_well.columns:
                        df_well[col] = np.nan

            # Need at least two classes overall in well
            if df_well['binary_label'].nunique() < 2:
                continue

            # --- Run time-aware classification for this well & feature set ---
            (
                y_true, y_pred, y_proba,
                selected_feats, X_test_enc, sv, bv,
                best_params
            ) = run_classification(
                df_well, 'binary_label', features, time_col='year_month'
            )

            if len(y_true) == 0:
                continue

            # Per-well test metrics
            acc = float(np.mean(np.array(y_true) == np.array(y_pred)))
            roc_auc = float(safe_roc_auc(y_true, y_proba))
            low_count = int(np.sum(np.array(y_true) == 0))
            high_count = int(np.sum(np.array(y_true) == 1))

            report_well = classification_report(
                y_true, y_pred,
                target_names=['Low risk', 'High risk'],
                output_dict=True, zero_division=0
            )
            prec_low = report_well['Low risk']['precision']
            rec_low = report_well['Low risk']['recall']
            f1_low = report_well['Low risk']['f1-score']
            prec_high = report_well['High risk']['precision']
            rec_high = report_well['High risk']['recall']
            f1_high = report_well['High risk']['f1-score']

            best_params_str = json.dumps(
                best_params if best_params is not None else DEFAULT_RF_PARAMS,
                sort_keys=True
            )

            # Store per-well results for this feature set
            well_results.append({
                'well_no': well_id,
                'accuracy': acc,
                'roc_auc': roc_auc,
                'low_risk_count': low_count,
                'high_risk_count': high_count,
                'precision_low': prec_low,
                'recall_low': rec_low,
                'f1_low': f1_low,
                'precision_high': prec_high,
                'recall_high': rec_high,
                'f1_high': f1_high,
                'best_params': best_params_str
            })

            # Combined table across feature sets
            per_well_accuracy_combined.append({
                'Feature Set': name,
                'well_no': well_id,
                'Accuracy': acc,
                'ROC AUC': roc_auc,
                'Low Risk Count': low_count,
                'High Risk Count': high_count,
                'Precision (Low)': prec_low,
                'Recall (Low)': rec_low,
                'F1 (Low)': f1_low,
                'Precision (High)': prec_high,
                'Recall (High)': rec_high,
                'F1 (High)': f1_high,
                'Best Params': best_params_str
            })

            # Track best configuration per well (NEW) using test metrics
            candidate = {
                'well_no': well_id,
                'feature_set': name,
                'accuracy': acc,
                'roc_auc': roc_auc,
                'score_value': _primary_score(roc_auc, acc)[0],
                'score_type': _primary_score(roc_auc, acc)[1],
                'best_params': best_params_str,
                'n_samples': int(len(y_true))
            }
            current = best_config_by_well.get(well_id)
            if current is None or _is_better(candidate, current):
                best_config_by_well[well_id] = candidate

            # Aggregate for overall (per feature set) confusion matrix and summary
            all_true.extend(y_true)
            all_preds.extend(y_pred)
            if y_proba is not None:
                all_probas.extend(y_proba)

            # -------------------- Save SHAP per (feature set, well) --------------------
            if sv is not None and X_test_enc is not None and selected_feats is not None and len(selected_feats) == X_test_enc.shape[1]:
                # Persist per-well dump
                np.savez_compressed(
                    os.path.join(per_set_dir, f"shap_well_{well_id}.npz"),
                    shap_values=sv,
                    base_values=bv if bv is not None else np.array([]),
                    data=X_test_enc.values,
                    feature_names=np.array(selected_feats)
                )
                # Collect for aggregation
                shap_collector[name]["X_list"].append(pd.DataFrame(X_test_enc.values, columns=selected_feats))
                shap_collector[name]["SV_list"].append(sv)
                shap_collector[name]["BV_list"].append(bv if bv is not None else np.array([]))
                shap_collector[name]["feature_names"] = selected_feats

        # Store per-well table for this feature set
        df_well_results = pd.DataFrame(well_results)
        per_well_tables[name] = df_well_results

        if not df_well_results.empty:
            out_path = f"per_well_results__{name}.csv"
            df_well_results.to_csv(out_path, index=False)
            logger.info(f"Saved per-well results for '{name}' -> {out_path}")

        # Confusion matrix (overall, on TEST across wells)
        if len(all_true) > 0:
            cm = confusion_matrix(all_true, all_preds, labels=[0, 1])
        else:
            cm = np.array([[0, 0], [0, 0]])
        confusion_matrices.append((name, cm))

        # Overall metrics (on TEST across wells)
        roc_auc_overall = safe_roc_auc(all_true, all_probas) if len(all_true) > 0 and len(all_probas) > 0 else np.nan
        low_count_overall = int(np.sum(np.array(all_true) == 0)) if len(all_true) > 0 else 0
        high_count_overall = int(np.sum(np.array(all_true) == 1)) if len(all_true) > 0 else 0

        if len(all_true) > 0:
            report = classification_report(
                all_true, all_preds, target_names=['Low risk', 'High risk'], output_dict=True, zero_division=0
            )
            accuracy_overall = report.get('accuracy', np.nan)
            prec_low = report['Low risk']['precision']
            rec_low = report['Low risk']['recall']
            f1_low = report['Low risk']['f1-score']
            prec_high = report['High risk']['precision']
            rec_high = report['High risk']['recall']
            f1_high = report['High risk']['f1-score']
        else:
            accuracy_overall = np.nan
            prec_low = rec_low = f1_low = np.nan
            prec_high = rec_high = f1_high = np.nan

        general_summary.append({
            'Feature Set': name,
            'Accuracy': accuracy_overall,
            'ROC AUC': roc_auc_overall,
            'Low Risk Count': low_count_overall,
            'High Risk Count': high_count_overall,
            'Precision (Low)': prec_low,
            'Recall (Low)': rec_low,
            'F1 (Low)': f1_low,
            'Precision (High)': prec_high,
            'Recall (High)': rec_high,
            'F1 (High)': f1_high,
        })

    # -------------------- Summary outputs --------------------
    df_general_summary = pd.DataFrame(general_summary)
    print("\n=== General Performance Summary (TEST across wells) ===")
    if not df_general_summary.empty:
        print(df_general_summary.round(3))
        df_general_summary.to_csv("general_summary.csv", index=False)
        logger.info("Saved -> general_summary.csv")
    else:
        print("(no results)")

    df_per_well_accuracy = pd.DataFrame(per_well_accuracy_combined)
    print("\n=== Accuracy Per Well (All Feature Sets) [TEST period only] ===")
    if not df_per_well_accuracy.empty:
        print(df_per_well_accuracy.round(3))
        df_per_well_accuracy.to_csv("per_well_accuracy.csv", index=False)
        logger.info("Saved -> per_well_accuracy.csv")
    else:
        print("(no results)")

    # Pivots (expanded to include the new metrics)
    if not df_per_well_accuracy.empty:
        pivot_metrics = [
            'Accuracy', 'ROC AUC',
            'Low Risk Count', 'High Risk Count',
            'Precision (Low)', 'Recall (Low)', 'F1 (Low)',
            'Precision (High)', 'Recall (High)', 'F1 (High)'
        ]
        for metric in pivot_metrics:
            if metric in df_per_well_accuracy.columns:
                pivot = df_per_well_accuracy.pivot(index='well_no', columns='Feature Set', values=metric)
                print(f"\n=== {metric} Per Well ===")
                print(pivot.round(3))

    # Confusion matrices
    for name, cm in confusion_matrices:
        print(f"\n=== Confusion Matrix for Feature Set: {name} (TEST across wells) ===")
        print(pd.DataFrame(cm, index=['True Low', 'True High'], columns=['Pred Low', 'Pred High']))

    # -------------------- NEW: Best configuration per well --------------------
    df_best_config = pd.DataFrame.from_dict(best_config_by_well, orient='index')
    df_best_config = df_best_config.sort_values('well_no').reset_index(drop=True)

    print("\n=== Best Performing Configuration Per Well (on TEST) ===")
    if not df_best_config.empty:
        display_cols = ['well_no', 'feature_set', 'score_type', 'score_value', 'accuracy', 'roc_auc', 'n_samples', 'best_params']
        print(df_best_config[display_cols].round(3))
        df_best_config.to_csv("best_config_per_well.csv", index=False)
        logger.info("Saved -> best_config_per_well.csv")
    else:
        print("(no results)")

    # -------------------- Aggregate & persist SHAP per feature set --------------------
    shap_aggregated = {}
    for fs_name, parts in shap_collector.items():

        X_list: list[pd.DataFrame] = parts["X_list"]
        SV_list: list[np.ndarray] = parts["SV_list"]
        BV_list: list[np.ndarray] = parts["BV_list"]
        if len(X_list) == 0 or len(SV_list) == 0:
            continue

        # Build a stable union of feature names preserving first-seen order
        union_cols = []
        seen = set()
        for X_df in X_list:
            for col in X_df.columns:
                if col not in seen:
                    union_cols.append(col)
                    seen.add(col)

        # Align each (X_df, SV) to the union by padding with zeros for missing features
        aligned_X = []
        aligned_SV = []
        aligned_BV = []
        for X_df, SV, BV in zip(X_list, SV_list, BV_list if len(BV_list) == len(SV_list) else [None]*len(SV_list)):
            # Coerce SV to 2D (n_samples, n_features_this_well)
            SV = np.asarray(SV)
            if SV.ndim == 3:
                if SV.shape[0] == X_df.shape[0]:
                    SV = SV[:, :, 1 if SV.shape[2] > 1 else 0]
                elif SV.shape[1] == X_df.shape[0]:
                    SV = SV[1 if SV.shape[0] > 1 else 0, :, :]
                else:
                    SV = SV.mean(axis=-1)
            elif SV.ndim == 1:
                SV = SV.reshape(-1, 1)

            # Reindex features to the union
            Xi = X_df.reindex(columns=union_cols, fill_value=np.nan)
            aligned_X.append(Xi)

            # Map SHAP columns (order == X_df.columns) into union matrix
            sv_pad = np.zeros((SV.shape[0], len(union_cols)), dtype=float)
            col_idx_map = {c: i for i, c in enumerate(union_cols)}
            for j, c in enumerate(X_df.columns):
                sv_pad[:, col_idx_map[c]] = SV[:, j]
            aligned_SV.append(sv_pad)

            if BV is not None and np.size(BV) > 0:
                aligned_BV.append(np.asarray(BV).ravel())
            else:
                aligned_BV.append(np.zeros(SV.shape[0]))

        X_concat = pd.concat(aligned_X, axis=0, ignore_index=True)
        SV_concat = np.vstack(aligned_SV)
        BV_concat = np.concatenate(aligned_BV)

        # Save aggregated dump
        out_npz = os.path.join("shap_values", f"{fs_name}__AGGREGATED.npz")
        np.savez_compressed(out_npz,
                            shap_values=SV_concat,
                            base_values=BV_concat,
                            data=X_concat.values,
                            feature_names=np.array(union_cols))
        logger.info(f"Saved aggregated SHAP -> {out_npz}")

        # Prepare Explanation for plotting API
        shap_expl = shap.Explanation(
            values=SV_concat,
            base_values=BV_concat,
            data=X_concat.values,
            feature_names=union_cols
        )
        shap_aggregated[fs_name] = {
            "explanation": shap_expl,
            "X_df": X_concat,
            "feature_names": union_cols
        }

    return {
        "general_summary": df_general_summary,
        "per_well_accuracy": df_per_well_accuracy,  # includes per-class metrics
        "per_well_tables": per_well_tables,
        "confusion_matrices": confusion_matrices,
        "best_config_per_well": df_best_config,
        "shap_aggregated": shap_aggregated
    }

if __name__ == "__main__":
    # Example (uncomment and replace with your data):
    df_work['year_month'] = ensure_datetime(df_work['year_month'])
    results = evaluate_feature_sets(df_work, feature_sets)
    pass

# Part III — Article figures and tables

The cells below are reporting/analytics only. They are intentionally placed **after both modeling stages** so the scientific workflow can be run first and the manuscript outputs generated afterward.

Outputs are ordered to follow the manuscript. Where the supplied notebooks do not contain the corresponding code, a placeholder cell is provided rather than inventing a different workflow.

Article: [Sinkhole risk forecasting in the Lithuania–Latvia Karst region using artificial intelligence](https://www.sciencedirect.com/science/article/pii/S2214581826002703)


## Article-only source data for environmental overview figures

The imputation pipeline intentionally restricts the seven-well modeling dataset to 2003 onward. The environmental overview figures in the article use the longer source record. To avoid changing the model input or any upstream state, this cell loads a **separate reporting-only copy** named `df_article_full`.

In [ ]:
# Reporting-only copy of the same master source table.
with zipfile.ZipFile(MASTER_DATA_ZIP) as z:
    master_member = _first_real_zip_member(z)
    with z.open(master_member) as f:
        df_article_full = pd.read_csv(f)

df_article_full['date'] = pd.to_datetime(df_article_full['date'])
print(f'Article-only source table: {df_article_full.shape[0]:,} rows × {df_article_full.shape[1]:,} columns')

## Figure 1 — Workflow of the data-processing and machine-learning pipeline

**Placeholder.** The manuscript presents the overall coupled groundwater-imputation and sinkhole-risk workflow.

In [ ]:
# TODO: paste/rebuild manuscript Figure 1 workflow code here.

## Figure 2 — Study design

**Placeholder.** Manuscript panels: (a) groundwater-level imputation and (b) sinkhole-risk machine-learning design.

In [ ]:
# TODO: paste/rebuild manuscript Figure 2 study-design diagram code here.

## Figure 3 — Transboundary karst-region map

Generated from the GIS layers and plotting logic supplied in `1_map(1).ipynb`. Panel (a) shows the regional setting with an inset locating the study area in Europe; panel (b) zooms to the main Lithuania–Latvia karst belt and labeled monitoring wells.

In [ ]:

# Figure 3 needs additional GIS files. Download them only here,
# so a map-source issue cannot block the main ML pipeline.
required_map_files = [
    REPO_ROOT / 'data' / 'map' / 'wells.csv',
    REPO_ROOT / 'data' / 'map' / 'karst_lithuania.geojson',
    REPO_ROOT / 'data' / 'map' / 'karst_latvia.geojson',
    REPO_ROOT / 'data' / 'map' / 'baltic_cities.geojson',
    REPO_ROOT / 'data' / 'map' / 'country_borders.geojson',
    REPO_ROOT / 'data' / 'map' / 'world.geojson',
    REPO_ROOT / 'data' / 'map' / 'elevation.geojson',
    REPO_ROOT / 'data' / 'map' / 'rivers.geojson',
    REPO_ROOT / 'data' / 'map' / 'karst_regions.geojson',
]

if not all(path.exists() for path in required_map_files):
    print('Figure 3 GIS inputs are missing; downloading map inputs...')
    map_result = subprocess.run(
        [
            sys.executable,
            str(REPO_ROOT / 'scripts' / 'download_data.py'),
            '--map',
        ],
        text=True,
        capture_output=True,
    )
    if map_result.stdout:
        print(map_result.stdout)
    if map_result.returncode != 0:
        if map_result.stderr:
            print(map_result.stderr)
        raise RuntimeError(
            'Figure 3 map-data download failed. '
            'See the detailed source error printed above.'
        )

# ============================================================
# FIGURE 3 — TRANSBOUNDARY KARST REGION MAP
# Local GitHub-repository inputs downloaded by scripts/download_data.py
# ============================================================

os.makedirs(FIGURES_DIR, exist_ok=True)

def add_north_arrow(ax, x, y, width, height):
    box = Rectangle(
        (x, y), width, height, transform=ax.transAxes,
        facecolor='white', edgecolor='black', linewidth=1.2, zorder=20,
    )
    ax.add_patch(box)
    arrow = FancyArrow(
        x + width / 2, y + height * 0.15, 0, height * 0.45,
        transform=ax.transAxes, width=width * 0.10,
        head_width=width * 0.35, head_length=height * 0.20,
        length_includes_head=True, color='black', zorder=21,
    )
    ax.add_patch(arrow)
    ax.text(
        x + width / 2, y + height * 0.85, 'N', transform=ax.transAxes,
        fontsize=12, fontweight='bold', ha='center', va='center', zorder=22,
    )

map_wells = pd.read_csv(DATA_MAP / 'wells.csv')
lt_karst = gpd.read_file(DATA_MAP / 'karst_lithuania.geojson').set_crs(epsg=3346, allow_override=True).to_crs(epsg=4326)
lv_karst = gpd.read_file(DATA_MAP / 'karst_latvia.geojson').set_crs(epsg=3346, allow_override=True).to_crs(epsg=4326)
baltic_cities = gpd.read_file(DATA_MAP / 'baltic_cities.geojson')
countries = gpd.read_file(DATA_MAP / 'country_borders.geojson')
world = gpd.read_file(DATA_MAP / 'world.geojson')
elev = gpd.read_file(DATA_MAP / 'elevation.geojson')
rivers = gpd.read_file(DATA_MAP / 'rivers.geojson')
karst_region = gpd.read_file(DATA_MAP / 'karst_regions.geojson').to_crs(epsg=4326)

map_geometry = [Point(xy) for xy in zip(map_wells['WGS_lon'], map_wells['WGS_lat'])]
gdf_wells = gpd.GeoDataFrame(map_wells, geometry=map_geometry, crs='EPSG:4326')

elev['Value'] = pd.to_numeric(elev['Value'], errors='coerce')
norm = mpl.colors.Normalize(vmin=elev['Value'].min(), vmax=elev['Value'].max())

hatch_map = {'96F': '//', '970': '\\\\', 'A3B': '--'}
hatch_labels = {
    '96F': 'Stabilized carbonate karst',
    '970': 'Stabilized gypsum karst',
    'A3B': 'Active gypsum karst',
}

fig, (ax_main, ax_zoom) = plt.subplots(1, 2, figsize=(14, 7))

# ---------------- Main regional panel ----------------
for entity, group in karst_region.groupby('EntityHand'):
    hatch = hatch_map.get(entity)
    group.plot(
        ax=ax_main,
        facecolor='none',
        edgecolor='black',
        hatch=hatch if hatch else None,
        zorder=1,
    )

lv_karst.plot(ax=ax_main, facecolor='none', edgecolor='red', linewidth=1.3, zorder=2)
lt_karst.plot(ax=ax_main, facecolor='none', edgecolor='red', linewidth=1.3, zorder=2)

elev.plot(
    ax=ax_main,
    column='Value',
    cmap='Spectral_r',
    norm=norm,
    markersize=25,
    linewidth=0.5,
    legend=True,
    legend_kwds={'label': 'Elevation (m)', 'orientation': 'vertical', 'shrink': 0.72},
    zorder=3,
)

gdf_wells.plot(ax=ax_main, color='blue', marker='^', markersize=24, label='Wells', zorder=4)
countries.plot(ax=ax_main, color='black', edgecolor='black', label='Country borders', zorder=3)
rivers.plot(ax=ax_main, color='deepskyblue', linewidth=1, label='Rivers', zorder=3)

population = 50000
cities_main = baltic_cities[
    (baltic_cities['country'].str.lower() != 'estonia')
    & (baltic_cities['population'] > population)
].copy()
cities_main.plot(ax=ax_main, color='black', markersize=9, label=f'Cities (population > {population})', zorder=5)

for x, y, label in zip(cities_main.geometry.x, cities_main.geometry.y, cities_main['city']):
    if label == 'Jūrmala':
        x -= 0.4
    ax_main.text(
        x, y, label, fontsize=8, ha='left', va='bottom',
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.85, boxstyle='round,pad=0.15'),
    )

for label, (x, y) in {
    'Lithuania': (22.5, 55.5),
    'Latvia': (25.0, 57.0),
    'Poland': (21.0, 54.2),
    'Belarus': (28.0, 55.5),
    'Baltic Sea': (20.8, 57.4),
}.items():
    ax_main.text(
        x, y, label, fontsize=11, weight='bold', color='dimgray',
        ha='center', va='center',
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, boxstyle='round,pad=0.2'),
    )

add_north_arrow(ax_main, x=0.01, y=0.84, width=0.055, height=0.13)

legend_extra = [
    Patch(facecolor='none', edgecolor='black', hatch=hatch_map[key], label=hatch_labels[key])
    for key in hatch_map
]
legend_extra.append(Patch(facecolor='none', edgecolor='red', linewidth=1.3, label='Modern active karst'))

handles, labels = ax_main.get_legend_handles_labels()
ax_main.legend(handles + legend_extra, labels + [p.get_label() for p in legend_extra], loc='lower right', fontsize=8)

ax_main.set_xlim(20, 29)
ax_main.set_ylim(54, 58)
ax_main.set_xlabel('Longitude')
ax_main.set_ylabel('Latitude')
ax_main.grid(True, linestyle='--', linewidth=0.5)
ax_main.text(0.01, 0.99, '(a)', transform=ax_main.transAxes, fontsize=14, fontweight='bold', va='top')

# Europe inset from the same map notebook source.
if world.crs is not None and world.crs.to_string() != 'EPSG:4326':
    world = world.to_crs(epsg=4326)

ax_inset = ax_main.inset_axes([0.63, 0.70, 0.34, 0.26])
world.plot(ax=ax_inset, color='lightgray', edgecolor='black', linewidth=0.3)
ax_inset.set_xlim(-25, 45)
ax_inset.set_ylim(35, 72)
ax_inset.add_patch(Rectangle((20, 54), 9, 4, linewidth=1.2, edgecolor='red', facecolor='none'))
ax_inset.set_xticks([])
ax_inset.set_yticks([])

# ---------------- Zoomed karst panel ----------------
for entity, group in karst_region.groupby('EntityHand'):
    hatch = hatch_map.get(entity)
    if hatch:
        group.plot(ax=ax_zoom, facecolor='none', edgecolor='black', hatch=hatch, zorder=1)

lv_karst.plot(ax=ax_zoom, facecolor='none', edgecolor='red', linewidth=1.5, zorder=2)
lt_karst.plot(ax=ax_zoom, facecolor='none', edgecolor='red', linewidth=1.5, zorder=2)
elev.plot(ax=ax_zoom, column='Value', cmap='Spectral_r', norm=norm, markersize=18, linewidth=0.5, zorder=2)
gdf_wells.plot(ax=ax_zoom, color='blue', marker='^', markersize=32, zorder=4)
countries.plot(ax=ax_zoom, color='black', edgecolor='black', zorder=3)
rivers.plot(ax=ax_zoom, color='deepskyblue', linewidth=1, label='Rivers', zorder=3)

population_zoom = 2000
cities_zoom = baltic_cities[
    (baltic_cities['country'].str.lower() != 'estonia')
    & (baltic_cities['population'] > population_zoom)
    & (baltic_cities['lng'].between(24, 25))
    & (baltic_cities['lat'].between(56, 56.5))
].copy()
cities_zoom.plot(
    ax=ax_zoom,
    color='black',
    markersize=10,
    label=f'Cities (population > {population_zoom})',
    zorder=5,
)

for x, y, label in zip(cities_zoom.geometry.x, cities_zoom.geometry.y, cities_zoom['city']):
    ax_zoom.text(
        x, y, label, fontsize=8, ha='left', va='bottom',
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.85, boxstyle='round,pad=0.15'),
    )

well_labels = {
    'LT_35995, LT_193\nLT_216, LT_218\nLT_220, LT_839': (24.73, 56.25),
    'LT_202': (24.6, 56.175),
    'LT_35994': (24.78, 56.17),
    'LT_35996': (24.25, 56.18),
    'LV_12225': (24.27, 56.37),
    'LV_22606\nLV_22610': (24.7, 56.45),
}

for label, (x, y) in well_labels.items():
    ax_zoom.text(
        x, y, label, fontsize=9, color='blue', ha='center', va='center',
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.9, boxstyle='round,pad=0.25'),
        zorder=6,
    )

for label, (x, y) in {'Lithuania': (24.5, 56.15), 'Latvia': (24.5, 56.45)}.items():
    ax_zoom.text(
        x, y, label, fontsize=11, weight='bold', color='dimgray', ha='center', va='center',
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, boxstyle='round,pad=0.2'),
    )

add_north_arrow(ax_zoom, x=0.01, y=0.82, width=0.07, height=0.15)
ax_zoom.set_xlim(24, 25)
ax_zoom.set_ylim(56, 56.5)
ax_zoom.set_xticks([24, 24.5, 25])
ax_zoom.set_yticks([56, 56.25, 56.5])
ax_zoom.set_xlabel('Longitude')
ax_zoom.set_ylabel('Latitude')
ax_zoom.grid(True, linestyle='--', linewidth=0.5)
ax_zoom.legend(frameon=True, loc='lower right', fontsize=8)
ax_zoom.text(0.01, 0.99, '(b)', transform=ax_zoom.transAxes, fontsize=14, fontweight='bold', va='top')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'figure_3_karst_region_map.jpeg', dpi=600, bbox_inches='tight')
plt.show()

## Figure 4 — Daily climate variables

Adapted from `4_level(1).ipynb`. The four panels show precipitation, temperature, actual evapotranspiration and potential evapotranspiration. Faint lines represent individual wells/locations and the darker line represents the per-date mean.

In [ ]:
# Prepare data
df_climate = df_article_full.copy()
df_climate['date'] = pd.to_datetime(df_climate['date'])

# Component configuration
components = [
    'precipitation_mm_eobs',
    'temperature_C_eobs',
    'actual_evapotranspiration_mm_gleam',
    'potential_evapotranspiration_mm_gleam'
]

# Custom settings per component
component_settings = {
    'precipitation_mm_eobs': {
        'color_light': 'lightblue',
        'color_dark': 'darkblue',
        'y_label': 'Precipitation (mm)'
    },
    'temperature_C_eobs': {
        'color_light': 'lightcoral',
        'color_dark': 'darkred',
        'y_label': 'Temperature (°C)'
    },
    'actual_evapotranspiration_mm_gleam': {
        'color_light': 'lightgreen',
        'color_dark': 'darkgreen',
        'y_label': 'Actual ET (mm)'
    },
    'potential_evapotranspiration_mm_gleam': {
        'color_light': '#d2b48c',  # tan
        'color_dark': '#8b4513',   # saddlebrown
        'y_label': 'Potential ET (mm)'
    }
}

# Line thickness controls
well_linewidth = 0.8
avg_linewidth = 0.5

# Compute per-date averages
df_avg_list = []
for comp in components:
    subset = df_climate[['date', comp]].dropna()
    avg = subset.groupby('date')[comp].mean().reset_index()
    avg['component'] = comp
    avg.rename(columns={comp: 'value'}, inplace=True)
    df_avg_list.append(avg)

df_avg_melted = pd.concat(df_avg_list, ignore_index=True)
df_avg_melted['well_no'] = 'Average'

# Melt original data
df_melted = df_climate.melt(
    id_vars=['date', 'well_no'],
    value_vars=components,
    var_name='component',
    value_name='value'
).dropna()

# Combine for plotting
df_combined = pd.concat([df_melted, df_avg_melted], ignore_index=True)

# Plotting
fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(8.27, 11.69), sharex=True)  # A4 portrait size in inches
label_letters = ['(a)', '(b)', '(c)', '(d)']

for ax, comp, label in zip(axes, components, label_letters):
    data = df_combined[df_combined['component'] == comp]
    well_data = data[data['well_no'] != 'Average']
    avg_data = data[data['well_no'] == 'Average']

    color_light = component_settings[comp]['color_light']
    color_dark = component_settings[comp]['color_dark']
    y_label = component_settings[comp]['y_label']

    # Plot well lines
    for _, group in well_data.groupby('well_no'):
        ax.plot(group['date'], group['value'], color=color_light, alpha=0.5, linewidth=well_linewidth)

    # Plot average line
    ax.plot(avg_data['date'], avg_data['value'], color=color_dark, linewidth=avg_linewidth, label='Average')

    # Labeling
    # ax.set_title(comp.replace('_', ' ').title(), fontsize=14)
    ax.set_ylabel(y_label, fontsize=12)
    # ax.legend(loc='upper right', fontsize=10)

    # Add subplot label in upper-left corner
    ax.text(0.01, 0.95, label, transform=ax.transAxes, fontsize=14, fontweight='bold', va='top', ha='left')

# X-axis label for the last plot
axes[-1].set_xlabel("Date", fontsize=12)

# Make tick labels larger
for ax in axes:
    ax.tick_params(axis='both', which='major', labelsize=10)

plt.tight_layout()
plt.savefig('figure_4_climate_features_timeseries.jpeg', dpi=600, bbox_inches='tight')
plt.show()


## Table 1 — Hydroclimatic summary statistics

Generated from the full reporting-only source record to remain consistent with the environmental overview rather than the 2003+ ML subset.

In [ ]:
components = [
    'precipitation_mm_eobs',
    'temperature_C_eobs',
    'actual_evapotranspiration_mm_gleam',
    'potential_evapotranspiration_mm_gleam',
]

table_1 = df_article_full[components].describe().T
table_1['median'] = df_article_full[components].median()

table_1 = table_1.rename(index={
    'precipitation_mm_eobs': 'Precipitation, mm E-OBS',
    'temperature_C_eobs': 'Temperature, °C E-OBS',
    'actual_evapotranspiration_mm_gleam': 'Actual Evapotranspiration, mm GLEAM',
    'potential_evapotranspiration_mm_gleam': 'Potential Evapotranspiration, mm GLEAM',
})

display(table_1.round(3))
table_1.to_csv('table_1_climate_summary.csv', float_format='%.3f')

## Figure 5 — Hydrogeological cross-section

**Placeholder.** Add the original cross-section/graphics workflow later.

In [ ]:
# TODO: paste the manuscript Figure 5 hydrogeological cross-section code here.

## Figure 6 — Monthly sinkhole occurrence

Manuscript structure: (a) monthly sinkhole counts through the study period and (b) totals by calendar month.

In [ ]:
# -------------------------
# Step 1: Clean & prepare
# -------------------------
df_sink_plot = df_sink[df_sink['date_event'].notna()].copy()
df_sink_plot['date_event'] = pd.to_datetime(df_sink_plot['date_event'])
# Keep a true monthly Period for grouping, then convert to Timestamp for plotting
df_sink_plot['year_month'] = df_sink_plot['date_event'].dt.to_period('M')

# -------------------------
# Step 2: Group monthly
# -------------------------
df_sink_plot_count_monthly = (
    df_sink_plot
    .groupby('year_month')
    .size()
    .rename('sink_count')
    .reset_index()
)

# Chronological order is better for a time series
df_sink_plot_count_monthly = df_sink_plot_count_monthly.sort_values('year_month')

# -------------------------
# Step 3: Filter from 2003+
# -------------------------
df_sink_plot_count_monthly = df_sink_plot_count_monthly[df_sink_plot_count_monthly['year_month'] >= pd.Period('2003-01', freq='M')]

# Convert Period -> Timestamp (month start) for plotting on a date axis
df_sink_plot_count_monthly['ym_ts'] = df_sink_plot_count_monthly['year_month'].dt.to_timestamp()

# -------------------------
# Step 4: Calendar-month totals (Jan–Dec)
# -------------------------
df_sink_plot_count_monthly['month_name'] = df_sink_plot_count_monthly['ym_ts'].dt.month_name()
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']

monthly_total = (
    df_sink_plot_count_monthly
    .groupby('month_name')['sink_count']
    .sum()
    .reindex(month_order)           # ensure calendar order
    .fillna(0)
)

# -------------------------
# Styling for visibility
# -------------------------
plt.style.use('default')
plt.rcParams.update({
    'figure.dpi': 200,
    'savefig.dpi': 600,
    'axes.titlesize': 20,
    'axes.labelsize': 20,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'legend.fontsize': 1,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
})

# -------------------------
# Step 5: Plot (two subplots)
# -------------------------
fig, axs = plt.subplots(2, 1, figsize=(20, 12), constrained_layout=True)
fig.patch.set_facecolor('white')

# (a) Monthly sink count time series
ax = axs[0]
ax.bar(
    df_sink_plot_count_monthly['ym_ts'],
    df_sink_plot_count_monthly['sink_count'],
    width=pd.Timedelta(days=24)    # ~1 month bar width for readability
)
# ax.set_title('Monthly Sinkhole Count (2003–present)')
ax.set_xlabel('Year–Month', fontsize=24)
ax.set_ylabel('Sinkhole Count', fontsize=24)

# Cleaner date axis
locator = mdates.AutoDateLocator()
formatter = mdates.ConciseDateFormatter(locator)
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(formatter)
ax.margins(x=0.01)
ax.tick_params(axis='x', rotation=0)
ax.text(0.01, 0.95, '(a)', transform=ax.transAxes, fontsize=18, fontweight='bold', va='top', ha='left')

# (b) Total occurrences by calendar month
ax2 = axs[1]
ax2.bar(monthly_total.index, monthly_total.values)
# ax2.set_title('Total Sinkhole Occurrences by Calendar Month (All Years ≥ 2003)')
ax2.set_xlabel('Month', fontsize=24)
ax2.set_ylabel('Total Sinkhole Count', fontsize=24)
ax2.tick_params(axis='x', rotation=0)
ax2.text(0.01, 0.95, '(b)', transform=ax2.transAxes, fontsize=24, fontweight='bold', va='top', ha='left')

# Finalize layout and save
plt.tight_layout()
plt.savefig('sinkhole_analysis.jpeg', dpi=600)
plt.show()

## Figure 7 — Regional TWS and GWS time series

Adapted from `4_level(1).ipynb`. TWS and GWS are averaged across available locations for each date and shown over the full storage-product period.

In [ ]:
# STEP 1: Prepare the DataFrame
df_tws = df_article_full.copy()
df_tws['date'] = pd.to_datetime(df_tws['date'])

# STEP 2: Define the components to plot
tws_components = {
    'tws_mm_tavg_gldas': {'color': 'royalblue',  'label': 'Total Water Storage (mm)'},
    'gws_mm_tavg_gldas': {'color': 'darkorange', 'label': 'Groundwater Storage (mm)'}
}

# --- Global styling for readability ---
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 600,
    'axes.titlesize': 20,
    'axes.labelsize': 16,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'legend.fontsize': 13,
    'lines.linewidth': 3,
})

# STEP 3: Create the plot
fig, ax = plt.subplots(figsize=(16, 9), constrained_layout=True)

for comp, settings in tws_components.items():
    subset = df_tws[['date', comp]].dropna()
    avg = subset.groupby('date', as_index=False)[comp].mean()
    ax.plot(avg['date'], avg[comp], label=settings['label'], color=settings['color'])

# Nice date formatting
locator = mdates.AutoDateLocator(minticks=6, maxticks=10)
formatter = mdates.ConciseDateFormatter(locator)
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(formatter)

# STEP 4: Final plot adjustments
ax.set_xlabel('Date', fontsize=24)
ax.set_ylabel('Storage Anomaly (mm)', fontsize=24)
# ax.set_title('Average Total and Groundwater Storage Over Time', pad=12)
ax.grid(True, which='major', linestyle='--', alpha=0.35)
ax.legend(frameon=True, ncol=1, fontsize=20, loc='lower right')
ax.margins(x=0.01)

plt.savefig('figure_7_tws_gws_timeseries.jpeg', dpi=600, bbox_inches='tight')
plt.show()

# 3.2 Groundwater-imputation results

## Figure 8 — SHAP attribution for groundwater-level imputation

The manuscript compares tuned Extra Trees SHAP summaries for (a) `no_seasonal` and (b) `with_seasonal`.

In [ ]:
SUBPLOT_LABEL_FS = 16

def _index_to_alpha_label(idx):
    letters = []
    while True:
        letters.append(chr(97 + idx % 26))
        idx = idx // 26 - 1
        if idx < 0:
            break
    return f"({''.join(reversed(letters))})"

def _concat_shap_for_feature_set(shap_values_all, feature_set_name, limit_wells=None):
    shap_arrays, feature_frames, used = [], [], 0
    for (_, fs), records in shap_values_all.items():
        if fs != feature_set_name:
            continue
        for rec in records:
            sv = rec['shap_values'].values if hasattr(rec['shap_values'], 'values') else rec['shap_values']
            shap_arrays.append(sv)
            feature_frames.append(rec['X_features'][rec['feature_names']])
            used += 1
            if limit_wells is not None and used >= limit_wells:
                break
        if limit_wells is not None and used >= limit_wells:
            break
    if not shap_arrays:
        return None, None, None
    features_df = pd.concat(feature_frames, ignore_index=True)
    return np.vstack(shap_arrays), features_df, list(features_df.columns)

def _apply_font_sizes_to_axes(fig, fonts):
    if not fonts:
        return
    tick_fs = fonts.get('tick')
    axis_label_fs = fonts.get('axis_label')
    cbar_tick_fs = fonts.get('cbar_tick', tick_fs)
    cbar_label_fs = fonts.get('cbar_label', axis_label_fs)
    for ax in fig.axes:
        if tick_fs is not None:
            ax.tick_params(axis='both', which='both', labelsize=tick_fs)
        if axis_label_fs is not None:
            ax.xaxis.label.set_size(axis_label_fs)
            ax.yaxis.label.set_size(axis_label_fs)
        if hasattr(ax, 'get_images') and len(ax.get_images()) == 0:
            if cbar_tick_fs is not None:
                ax.tick_params(axis='both', which='both', labelsize=cbar_tick_fs)
            if cbar_label_fs is not None:
                ax.yaxis.label.set_size(cbar_label_fs)

def _rc_updates_from_fonts(fonts=None, font_scale=1.0):
    fonts = fonts or {}
    scaled = lambda key: None if fonts.get(key) is None else float(fonts[key]) * float(font_scale)
    rc = {}
    mapping = {
        'base': ['font.size'],
        'title': ['axes.titlesize'],
        'axis_label': ['axes.labelsize'],
        'tick': ['xtick.labelsize', 'ytick.labelsize'],
        'legend': ['legend.fontsize'],
        'suptitle': ['figure.titlesize'],
    }
    for key, rc_keys in mapping.items():
        value = scaled(key)
        if value is not None:
            for rc_key in rc_keys:
                rc[rc_key] = value
    return rc

def _render_beeswarm_to_png_bytes(
    shap_mat, features_df, feature_names, title, max_display=20, rng=None,
    panel_dpi=600, fonts=None, font_scale=1.0, xlim=None
):
    rc_updates = _rc_updates_from_fonts(fonts, font_scale)
    with plt.rc_context(rc=rc_updates):
        try:
            shap.summary_plot(
                shap_mat, features=features_df, feature_names=feature_names,
                max_display=max_display, show=False, rng=rng
            )
        except TypeError:
            shap.summary_plot(
                shap_mat, features=features_df, feature_names=feature_names,
                max_display=max_display, show=False
            )

        fig = plt.gcf()
        if xlim is not None and fig.axes:
            try:
                main_ax = next(ax for ax in fig.axes if getattr(ax, 'collections', []))
            except StopIteration:
                main_ax = fig.axes[0]
            main_ax.set_xlim(xlim)

        if fig.axes:
            title_fs = fonts.get('title') * float(font_scale) if fonts and 'title' in fonts else None
            fig.axes[0].set_title(title, pad=10, fontsize=title_fs)

        _apply_font_sizes_to_axes(
            fig,
            {k: (v * float(font_scale) if v is not None else None) for k, v in (fonts or {}).items()},
        )

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=panel_dpi, bbox_inches='tight')
        plt.close(fig)

    buf.seek(0)
    return buf.getvalue()

def plot_stacked_beeswarms(
    shap_values_all, feature_sets, outdir='shap_plots', max_display=20, limit_wells=None,
    composite_fname='beeswarm_stacked.jpg', show=True, rng_seed=None, rng=None,
    final_dpi=600, panel_dpi=600, fonts=None, font_scale=1.0,
    subplot_label_fs=None, xlim=None
):
    os.makedirs(outdir, exist_ok=True)
    if rng is None and rng_seed is not None:
        rng = np.random.default_rng(rng_seed)

    child_rngs = None
    if rng is not None:
        state = rng.bit_generator.state
        try:
            base_seed = state['state']['state']
        except Exception:
            base_seed = abs(hash(repr(state))) % (2**32 - 1)
        child_rngs = [np.random.default_rng(s) for s in np.random.SeedSequence(base_seed).spawn(len(feature_sets))]

    png_images = []
    for i, fs_name in enumerate(feature_sets):
        shap_mat, features_df, feature_names = _concat_shap_for_feature_set(
            shap_values_all, fs_name, limit_wells=limit_wells
        )
        if shap_mat is None:
            print(f"[SHAP] No SHAP data for feature set '{fs_name}'. Skipping.")
            continue

        png_bytes = _render_beeswarm_to_png_bytes(
            shap_mat, features_df, feature_names, fs_name,
            max_display=max_display,
            rng=child_rngs[i] if child_rngs else rng,
            panel_dpi=panel_dpi,
            fonts=fonts,
            font_scale=font_scale,
            xlim=xlim,
        )
        png_images.append(plt.imread(io.BytesIO(png_bytes), format='png'))

    if not png_images:
        print('[SHAP] No SHAP data found.')
        return

    fig_height = sum(img.shape[0] for img in png_images) / float(panel_dpi)
    fig_width = max(img.shape[1] for img in png_images) / float(panel_dpi)
    fig, axes = plt.subplots(len(png_images), 1, figsize=(fig_width, fig_height), constrained_layout=True)
    axes = [axes] if len(png_images) == 1 else axes

    if subplot_label_fs is None:
        subplot_label_fs = (
            fonts['subplot_label'] * float(font_scale)
            if fonts and 'subplot_label' in fonts
            else SUBPLOT_LABEL_FS * float(font_scale)
        )

    for idx, (ax, img) in enumerate(zip(axes, png_images)):
        ax.imshow(img, zorder=0)
        ax.axis('off')
        ax.text(
            0.015, 0.985, _index_to_alpha_label(idx), transform=ax.transAxes,
            fontsize=subplot_label_fs, fontweight='bold', va='top', ha='left', zorder=5,
            bbox=dict(facecolor='white', alpha=0.85, edgecolor='none', boxstyle='round,pad=0.15'),
        )

    outpath = os.path.join(outdir, composite_fname)
    fig.savefig(outpath, format='jpeg', dpi=final_dpi, bbox_inches='tight')
    if show:
        plt.show()
    plt.close(fig)
    print(f'[SHAP] Saved stacked beeswarm JPEG to: {outpath}')

compare_feature_sets = ['no_seasonal', 'with_seasonal']
available_fs = sorted({fs for _, fs in shap_values_all.keys()})
selected = [fs for fs in compare_feature_sets if fs in available_fs]
if len(selected) < 2:
    selected = available_fs[:2]

if len(selected) < 2:
    print('Need at least two feature sets with SHAP data.')
else:
    my_fonts = {
        'base': 11, 'title': 14, 'axis_label': 14, 'tick': 13,
        'legend': 11, 'cbar_tick': 13, 'cbar_label': 18,
        'suptitle': 16, 'subplot_label': 16,
    }
    plot_stacked_beeswarms(
        shap_values_all, feature_sets=selected, outdir='shap_plots', max_display=20,
        limit_wells=None, composite_fname='beeswarm_stacked.jpg', show=True,
        rng_seed=42, final_dpi=600, panel_dpi=600, fonts=my_fonts,
        font_scale=1.3, xlim=(-1, 1),
    )

## Table 2 — Groundwater-level imputation performance by well

The table is generated directly from the reproduced `summary` object created during the imputation model run.

In [ ]:
table_2 = summary.copy()
table_2 = table_2.rename(columns={
    'CV_R2 mean': 'CV R² mean',
    'CV_R2 std': 'CV R² std',
    'CV_MAE mean': 'CV MAE mean',
    'CV_MAE std': 'CV MAE std',
    'feature_set_name': 'Feature set',
    'best_params': 'Best Hyperparameters',
})
display(table_2)

## Figure 9 — Groundwater-level noise versus cross-validated R²

Noise is represented by the standard deviation of first differences in the standardized groundwater-level signal and compared with mean CV R².

In [ ]:
df_sorted = df_imputed_vals.sort_values(['well_no', 'date']).copy()
df_sorted['gw_level_zscore'] = df_sorted.groupby('well_no')['gw_level_m_asl'].transform(
    lambda x: (x - x.mean()) / x.std()
)
noise_summary = df_sorted.groupby('well_no')['gw_level_zscore'].agg(std_dev='std', mean='mean').reset_index()
noise_summary['cv'] = noise_summary['std_dev'] / noise_summary['mean']
df_sorted['first_diff'] = df_sorted.groupby('well_no')['gw_level_zscore'].diff()
first_diff_std = df_sorted.groupby('well_no')['first_diff'].std().reset_index(name='std_first_diff')
noise_summary = noise_summary.merge(first_diff_std, on='well_no')
noise_summary

plot_df = summary[summary['well_no'] != 'AVERAGE'].copy().merge(noise_summary, on='well_no', how='left')

LABEL_FS, TICK_FS = 16, 14
plt.figure(figsize=(8, 6))
ax = sns.regplot(
    data=plot_df, x='std_first_diff', y='CV_R2 mean',
    scatter_kws={'s': 50, 'alpha': 0.7},
    line_kws={'linewidth': 2.2},
)
ax.set_xlabel('Groundwater-level noise (std of first differences)', fontsize=LABEL_FS)
ax.set_ylabel('Mean CV R²', fontsize=LABEL_FS)
ax.tick_params(axis='both', labelsize=TICK_FS)
ax.grid(True, which='major', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('gw_level_noise.jpeg', dpi=600, bbox_inches='tight', pad_inches=0.2)
plt.show()

## Figure 10 — Groundwater-level reconstruction by monitoring well

Observed groundwater levels are shown together with reconstructed values and the stored validation intervals.

In [ ]:
# ============================================================
# FIGURE 10 — GROUNDWATER-LEVEL IMPUTATION TIME SERIES
# ============================================================

A4_W_IN, A4_H_IN, DPI = 8.27, 11.69, 300

plt.rcParams.update({
    'figure.dpi': DPI,
    'savefig.dpi': DPI,
    'axes.titlesize': 14,
    'axes.labelsize': 16,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 13,
})

# df_work is reused later by the monthly risk workflow.
# df_merged is the preserved DAILY output of the imputation stage.
daily_df = df_merged.copy()
daily_df['date'] = pd.to_datetime(daily_df['date'])

required_cols = {'well_no', 'date', target_col}
missing_cols = required_cols - set(daily_df.columns)
if missing_cols:
    raise KeyError(f'Missing required columns in daily_df: {missing_cols}')

plots = []

for (model_name, feature_set_name), data_list in imputed_all_models.items():
    if not data_list:
        continue

    df_imputed = pd.concat(data_list, ignore_index=True)
    df_imputed['date'] = pd.to_datetime(df_imputed['date'])
    df_imputed = df_imputed.sort_values(['well_no', 'date'])

    for well_no in df_imputed['well_no'].unique():
        imputed = df_imputed[df_imputed['well_no'] == well_no].sort_values('date').copy()
        actual = daily_df[daily_df['well_no'] == well_no][['date', target_col]].sort_values('date').copy()

        observed = actual.dropna(subset=[target_col]).copy()
        missing_dates = actual.loc[actual[target_col].isna(), 'date']

        imputed_only = imputed[
            imputed['date'].isin(missing_dates) & imputed[target_col].notna()
        ][['date', target_col]].copy()

        observed['gap'] = observed['date'].diff().dt.days.fillna(1)
        observed['segment'] = (observed['gap'] > 1).cumsum()

        masked_segments = cv_masked_intervals.get(
            (well_no, model_name, feature_set_name),
            [],
        )

        plots.append({
            'well_no': well_no,
            'observed': observed,
            'imputed': imputed_only,
            'masked_segments': masked_segments,
        })

if not plots:
    raise ValueError('No groundwater-level time series available to plot.')

fig, axes = plt.subplots(
    len(plots),
    1,
    figsize=(A4_W_IN, A4_H_IN),
    sharex=True,
    squeeze=False,
)
axes = axes.ravel()

locator = mdates.AutoDateLocator()
formatter = mdates.ConciseDateFormatter(locator)
legend_items = OrderedDict()

for idx, (ax, item) in enumerate(zip(axes, plots)):
    well_no = item['well_no']
    observed = item['observed']
    imputed_only = item['imputed']
    masked_segments = item['masked_segments']

    for _, segment in observed.groupby('segment'):
        line = ax.plot(
            segment['date'],
            segment[target_col],
            color='black',
            linewidth=2.0,
        )[0]
        legend_items.setdefault('Observed', line)

    if not imputed_only.empty:
        scatter = ax.scatter(
            imputed_only['date'],
            imputed_only[target_col],
            s=18,
            color='tab:blue',
            zorder=3,
        )
        legend_items.setdefault('Imputed', scatter)

    for start_date, end_date in masked_segments:
        span = ax.axvspan(
            start_date,
            end_date,
            color='red',
            alpha=0.20,
            zorder=0,
        )
        legend_items.setdefault('CV Masked Interval', span)

    ax.text(
        0.01, 0.97, f'({chr(97 + idx)})',
        transform=ax.transAxes,
        fontsize=16,
        fontweight='bold',
        va='top',
        ha='left',
    )
    ax.set_title(f'Well {well_no}', pad=5)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    ax.grid(axis='x', linestyle=':', alpha=0.4)
    ax.margins(x=0.01)

    if idx < len(plots) - 1:
        ax.tick_params(axis='x', labelbottom=False)

axes[-1].xaxis.set_major_locator(locator)
axes[-1].xaxis.set_major_formatter(formatter)
axes[-1].set_xlabel('Date')

fig.supylabel('Groundwater level (m ASL)', x=0.01, fontsize=16)

fig.legend(
    list(legend_items.values()),
    list(legend_items.keys()),
    loc='lower center',
    ncol=3,
    frameon=True,
    bbox_to_anchor=(0.5, 0.015),
)

fig.tight_layout(rect=(0.03, 0.07, 1, 1))
fig.savefig(
    'stacked_imputation_timeseries.jpeg',
    dpi=DPI,
    bbox_inches='tight',
    pad_inches=0.2,
)
plt.show()

# 3.3 Sinkhole-risk results

## Figure 11 — Risk-threshold comparison

The original threshold diagnostic compares IQR, z-score and empirical 90th-percentile criteria. The operational model itself continues to use the reproduced fixed threshold defined in Part II.

In [ ]:
# Filter data
filtered_df = df_work[df_work['well_no'] == 'LT_220']

# Group by year_month and calculate mean sink_count per month
monthly_avg = filtered_df.groupby('year_month')['sink_count'].mean()

# Overall average and median of monthly averages
overall_avg = monthly_avg.mean()
overall_median = monthly_avg.median()

print(f"Average monthly sink_count: {overall_avg:.2f}")
print(f"Median monthly sink_count: {overall_median:.2f}")

# --- Threshold Evaluation ---

# IQR Method
Q1 = monthly_avg.quantile(0.25)
Q3 = monthly_avg.quantile(0.75)
IQR = Q3 - Q1
iqr_threshold = Q3 + 1.5 * IQR

# Z-Score Method
mean = monthly_avg.mean()
std = monthly_avg.std()
zscore_threshold = mean + 2 * std

# Percentile Method (90th percentile)
percentile_threshold = monthly_avg.quantile(0.90)

# K-Means Clustering (2 clusters: low and high risk)
kmeans = KMeans(n_clusters=2, random_state=0)
monthly_avg_reshaped = monthly_avg.values.reshape(-1, 1)
kmeans.fit(monthly_avg_reshaped)
labels = kmeans.labels_
cluster_centers = kmeans.cluster_centers_.flatten()

# Identify which cluster is high risk
high_risk_cluster = labels[np.argmax(cluster_centers)]
monthly_avg_clustered = pd.DataFrame({
    'year_month': monthly_avg.index,
    'avg_sink_count': monthly_avg.values,
    'risk_cluster': labels
})
monthly_avg_clustered['risk_label'] = monthly_avg_clustered['risk_cluster'].apply(
    lambda x: 'High Risk' if x == high_risk_cluster else 'Low Risk'
)

# --- Print thresholds ---
print(f"IQR-based High Risk Threshold: {iqr_threshold:.2f}")
print(f"Z-Score-based High Risk Threshold: {zscore_threshold:.2f}")
print(f"90th Percentile Threshold: {percentile_threshold:.2f}")
print("K-Means Clustering: Cluster centers =", cluster_centers)

# --- Optional: Visualization ---
plt.figure(figsize=(12, 6))
plt.hist(monthly_avg, bins=20, edgecolor='black', alpha=0.6)
plt.axvline(iqr_threshold, color='red', linestyle='--', label='IQR Threshold')
plt.axvline(zscore_threshold, color='blue', linestyle='--', label='Z-Score Threshold')
plt.axvline(percentile_threshold, color='green', linestyle='--', label='90th Percentile')
plt.xlabel('Average Sinkhole Count per Month')
plt.ylabel('Frequency')
plt.legend(fontsize=16, title="Thresholds", title_fontsize=16)
plt.grid(True)
plt.savefig("risk_threshold_estimation.jpeg", dpi=600)
plt.show()

## Figure 12 — SHAP summaries: groundwater, climate, GS and CS

Manuscript panel order:
(a) Groundwater, (b) Climate, (c) Groundwater + Season (GS), and (d) Climate + Season (CS).

In [ ]:
def plot_beeswarms_for_feature_sets(
    shap_agg: dict,
    selected_feature_sets: list[str] | None = None,  # choose which feature sets to plot (order respected)
    max_display: int = 15,
    save_path: str | None = "shap_beeswarms_grid.jpeg",
    ncols: int = 2,
    use_constrained_layout: bool = True,
    colorbar_mode: str = "last",           # {"all","first","last","none"}

    # ---- spacing controls ----
    row_spacing: float | None = None,      # vertical space between rows (hspace)
    col_spacing: float | None = None,      # horizontal space between columns (wspace)
    edge_pad_h: float | None = None,       # top/bottom padding around the grid (inches; CL only)
    edge_pad_w: float | None = None,       # left/right padding around the grid (inches; CL only)

    *,
    # ---- fonts & sizing ----
    font_scale: float = 1.0,               # multiply all sizes below
    title_size: int = 12,                  # subplot title font size (before scaling)
    title_pad: float = 6.0,                # distance from title to axes, in points
    tick_size: int = 14,                   # x-axis tick labels
    panel_label_size: int = 16,            # "(a)" panel letters
    cb_tick_size: int | None = None,       # colorbar ticks (defaults to tick_size if None)
    cb_label_size: int | None = None,      # colorbar label (defaults to title_size if None)

    # ---- x-axis controls ----
    x_label_text: str = "SHAP value (impact on model output)",
    x_label_size: int | None = None,       # defaults to title_size if None (after scaling)
    x_label_pad: float = 8.0,              # distance from x label to axis, in points
    show_x_label: bool = True,             # toggle to show/hide the x-axis label
    xlim: tuple[float, float] | None = (-1.0, 1.0),  # fix x-axis; set to None to auto

    # ---- aspect / page-shape controls (NEW) ----
    target_aspect: float | None = 0.707,   # ensure width/height >= 0.707 (A4 portrait); set None to disable
    enforce_exact_aspect: bool = False,    # if True, width = height * target_aspect (exact A4-portrait shape)

    # ---- subplot titles & panel letters ----
    show_titles: bool = True,              # show feature set name as subplot title
    panel_label_offset: tuple[float, float] = (-0.40, 0.98),  # axes coords (x,y) for "(a)"

    # ---- feature-name (y-axis) toggle ----
    show_feature_names: bool = True,
    feature_name_size: int | None = None,  # y-axis labels; defaults to tick_size if None

    # ---- output ----
    dpi: int = 600,                        # save resolution
):
    """
    Plot a grid of SHAP beeswarm plots from a mapping:
      {feature_set_name: {"explanation": shap.Explanation, ...}, ...}

    Highlights
    ----------
    • Select a subset via `selected_feature_sets` (order respected).
    • Robust axes handling for 1×N, N×1, and 1×1 layouts.
    • Panel letters "(a)", "(b)", ... per subplot.
    • Font scaling and detailed spacing controls.
    • Per-subplot colorbar control via `colorbar_mode` = {"all","first","last","none"}.
    • Full control of the x-axis label.
    • `xlim` to fix the x-axis range (defaults to [-1, 1]).
    • NEW: `target_aspect` widens the figure toward A4 portrait (width/height ≈ 0.707).
            Set `enforce_exact_aspect=True` for an exact A4-portrait shape.
    """
    import math
    import warnings
    import numpy as np
    import matplotlib.pyplot as plt
    import shap

    # ---- basic checks ----
    if not shap_agg:
        warnings.warn("shap_agg is empty. Nothing to plot.")
        return

    if colorbar_mode not in {"all", "first", "last", "none"}:
        raise ValueError("colorbar_mode must be one of {'all','first','last','none'}")

    # ---- resolve font sizes with scaling ----
    title_size = title_size * font_scale
    tick_size = tick_size * font_scale
    panel_label_size = panel_label_size * font_scale
    cb_tick_size = tick_size if cb_tick_size is None else cb_tick_size * font_scale
    cb_label_size = title_size if cb_label_size is None else cb_label_size * font_scale
    feature_name_size = tick_size if feature_name_size is None else feature_name_size * font_scale
    x_label_size = title_size if x_label_size is None else x_label_size * font_scale

    # ---- choose which feature sets to plot (and in what order) ----
    if selected_feature_sets is None:
        fs_names = list(shap_agg.keys())
    else:
        fs_names = [name for name in selected_feature_sets if name in shap_agg]
        missing = [name for name in selected_feature_sets if name not in shap_agg]
        if missing:
            warnings.warn(f"Ignoring unknown feature sets: {missing}")

    if not fs_names:
        warnings.warn("No valid feature sets to plot after filtering. Nothing to do.")
        return

    # ---- grid geometry ----
    n = len(fs_names)
    ncols = max(1, int(ncols))
    nrows = int(math.ceil(n / ncols))

    # ---- base figure size heuristic ----
    fig_width = 14 if ncols == 2 else 10 + 3 * (ncols - 1)
    fig_height = max(4.8, 5.2 * nrows)

    # ---- widen toward A4 portrait if requested ----
    if target_aspect is not None and target_aspect > 0:
        current_aspect = fig_width / fig_height
        desired_width = fig_height * target_aspect
        if enforce_exact_aspect:
            fig_width = desired_width
        else:
            # Ensure at least the target aspect (only widens if currently too narrow)
            fig_width = max(fig_width, desired_width)

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(fig_width, fig_height),
        constrained_layout=use_constrained_layout,
    )

    # Normalize axes to a flat 1-D array (handles 1×N, N×1, 1×1)
    axes_flat = axes.ravel() if isinstance(axes, np.ndarray) else np.array([axes], dtype=object)
    orig_size = fig.get_size_inches().copy()  # remember intended size

    # ---- apply spacing preferences (constrained layout only) ----
    if use_constrained_layout:
        pads = {}
        if edge_pad_w is not None: pads["w_pad"] = edge_pad_w
        if edge_pad_h is not None: pads["h_pad"] = edge_pad_h
        if col_spacing is not None: pads["wspace"] = col_spacing
        if row_spacing is not None: pads["hspace"] = row_spacing
        if pads:
            try:
                fig.set_constrained_layout_pads(**pads)
            except Exception:
                pass

    # ---- helpers ----
    def _panel_tag(i: int) -> str:
        # (a), (b), ..., (z), (aa), (ab), ...
        letters = []
        k = i
        while True:
            k, rem = divmod(k, 26)
            letters.append(chr(97 + rem))
            if k == 0:
                break
            k -= 1
        return f"({''.join(reversed(letters))})"

    def _show_colorbar_for(idx: int) -> bool:
        return (
            colorbar_mode == "all"
            or (colorbar_mode == "first" and idx == 0)
            or (colorbar_mode == "last" and idx == n - 1)
        )

    # ---- plotting loop ----
    for idx, fs_name in enumerate(fs_names):
        ax = axes_flat[idx]
        plt.sca(ax)

        # Track axes before SHAP draw (to find any new colorbar axes)
        axes_before = set(map(id, fig.axes))

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            # Support older/newer SHAP signatures
            try:
                shap.plots.beeswarm(
                    shap_agg[fs_name]["explanation"],
                    max_display=max_display,
                    show=False,
                    color_bar=_show_colorbar_for(idx),
                )
            except TypeError:
                shap.plots.beeswarm(
                    shap_agg[fs_name]["explanation"],
                    max_display=max_display,
                    show=False,
                )

        # SHAP sometimes resizes the figure; undo that
        fig.set_size_inches(orig_size, forward=True)

        # ---- style axes ----
        ax.tick_params(axis="x", labelsize=tick_size)

        if xlim is not None:
            try:
                ax.set_xlim(xlim[0], xlim[1])
            except Exception:
                pass

        if show_x_label:
            try:
                ax.set_xlabel(x_label_text, fontsize=x_label_size, labelpad=x_label_pad)
            except Exception:
                ax.set_xlabel(x_label_text)
                try:
                    ax.xaxis.label.set_size(x_label_size)
                except Exception:
                    pass
        else:
            ax.set_xlabel("")

        ax.tick_params(axis="y", labelsize=feature_name_size, labelleft=show_feature_names)
        if not show_feature_names:
            ax.tick_params(axis="y", which="both", length=0)
            try:
                ax.set_yticklabels([])
            except Exception:
                pass

        if show_titles:
            ax.set_title(fs_name, fontsize=title_size, pad=title_pad)

        ax.text(
            panel_label_offset[0],
            panel_label_offset[1],
            _panel_tag(idx),
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=panel_label_size,
            fontweight="bold",
        )

        # Style any new axes (e.g., colorbars)
        new_axes = [a for a in fig.axes if id(a) not in axes_before]
        for cb_ax in new_axes:
            if cb_ax is ax:
                continue
            try:
                cb_ax.tick_params(labelsize=cb_tick_size)
                if getattr(cb_ax.yaxis, "label", None):
                    cb_ax.yaxis.label.set_size(cb_label_size)
                if getattr(cb_ax.xaxis, "label", None):
                    cb_ax.xaxis.label.set_size(cb_label_size)
            except Exception:
                pass

    # ---- hide any unused axes ----
    for idx in range(n, nrows * ncols):
        axes_flat[idx].axis("off")

    # ---- layout & save ----
    if not use_constrained_layout:
        try:
            fig.tight_layout(rect=(0, 0, 1, 0.97))
        except Exception:
            pass
        fig.subplots_adjust(
            top=0.93,
            left=0.08,
            right=0.98,
            wspace=0.28 if col_spacing is None else col_spacing,
            hspace=0.34 if row_spacing is None else row_spacing,
        )

    if save_path:
        fig.canvas.draw_idle()
        if use_constrained_layout:
            fig.savefig(save_path, dpi=dpi)  # with CL, avoid bbox_inches
        else:
            fig.savefig(save_path, dpi=dpi, bbox_inches="tight")

    plt.show()

In [ ]:
plot_beeswarms_for_feature_sets(
    results["shap_aggregated"],
    selected_feature_sets=["Groundwater Features", "Climatic Features", "GS Combined Features",  "CS Combined Features"],
    max_display=15,
    save_path="shap_beeswarms_sinkholes_1.jpeg",
    ncols=1,
    use_constrained_layout=True,
    colorbar_mode="all",
    # spacing
    row_spacing=0.2,
    edge_pad_h=0.1,
    edge_pad_w=0.5,
    # fonts & labels
    font_scale=1.15,         # bump everything
    title_size=16,           # subplot titles
    title_pad=16.0,
    tick_size=15,            # x-axis ticks
    feature_name_size=16,    # y-axis feature names
    panel_label_size=16,     # "(a)" labels
    cb_tick_size=15,         # colorbar tick labels
    cb_label_size=15,        # colorbar title
    x_label_text="SHAP value",
    x_label_size=16,
    show_titles=True,
    panel_label_offset=(-1.0, 0.98),
    show_feature_names=True
)

## Figure 13 — SHAP summaries: TWS, GWS, GWS-seasonal and CGS

Manuscript panel order:
(a) TWS, (b) GWS, (c) GWS + seasonal terms, and (d) combined climate–groundwater–seasonal (CGS).

In [ ]:
plot_beeswarms_for_feature_sets(
    results["shap_aggregated"],
    selected_feature_sets=["TWS Features", "GWS Features", "GWS-Seasonal-Features",  "CGS Combined Features"],
    max_display=15,
    save_path="shap_beeswarms_sinkholes_2.jpeg",
    ncols=1,
    use_constrained_layout=True,
    colorbar_mode="all",
    # spacing
    row_spacing=0.2,
    edge_pad_h=0.1,
    edge_pad_w=2.0,
    # fonts & labels
    font_scale=1.15,         # bump everything
    title_size=16,           # subplot titles
    title_pad=16.0,
    tick_size=15,            # x-axis ticks
    feature_name_size=14,    # y-axis feature names
    panel_label_size=16,     # "(a)" labels
    cb_tick_size=15,         # colorbar tick labels
    cb_label_size=15,        # colorbar title
    x_label_text="SHAP value",
    x_label_size=16,
    show_titles=True,
    panel_label_offset=(-1.0, 0.98),
    show_feature_names=True
)

## Figure 14 — Groundwater level and sinkhole activity by well

The manuscript compares monthly groundwater level and sinkhole counts for the seven monitoring wells, with the risk threshold and ±30-day windows around identified groundwater maxima.

In [ ]:
# -------------------- CONFIG --------------------
# Print-ready sizing (A4 portrait) and larger fonts
A4_W_IN, A4_H_IN = 8, 12   # inches (approx A4 portrait)
DPI = 600                  # print quality

plt.rcParams.update({
    "figure.dpi": DPI,
    "savefig.dpi": DPI,
    "axes.titlesize": 14,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 13,
})

# -------------------- INPUTS --------------------
# Expect these to exist:
# - df_work: DataFrame with columns ['well_no','year_month','gw_level_m_asl_imputed','sink_count', ('gw_level_near_peak' optional)]
# - threshold: numeric value for sinkhole risk threshold

# -------------------- PREP --------------------
wells   = sorted(df_work['well_no'].unique())
n_plots = len(wells)
if n_plots == 0:
    raise ValueError("No wells found to plot.")

# Figure with stacked subplots and shared x-axis
fig, axes = plt.subplots(n_plots, 1, figsize=(A4_W_IN, A4_H_IN), sharex=True)
if n_plots == 1:
    axes = [axes]

# Date formatting for the shared x-axis
locator   = mdates.AutoDateLocator()
formatter = mdates.ConciseDateFormatter(locator)

# Collect legend entries once, then show a single legend at the bottom
legend_pairs = []  # list of (handle, label)

# -------------------- PLOTTING --------------------
for idx, (ax1, well_id) in enumerate(zip(axes, wells)):
    df_well = (
        df_work.loc[df_work['well_no'] == well_id]
        .sort_values('year_month')
        .copy()
    )

    # LEFT axis: GW level time series (blue)
    h_gw = ax1.plot(
        df_well['year_month'],
        df_well['gw_level_m_asl_imputed'],
        linewidth=2.0, label='GW Level (m ASL)', color='tab:blue'
    )
    # Y padding (±10%)
    gw_min = df_well['gw_level_m_asl_imputed'].min()
    gw_max = df_well['gw_level_m_asl_imputed'].max()
    gw_rng = (gw_max - gw_min) if pd.notna(gw_max) and pd.notna(gw_min) else 1.0
    ax1.set_ylim(gw_min - 0.10 * gw_rng, gw_max + 0.10 * gw_rng)

    # Style left y-axis blue (no per-axes y-labels; figure-level label added later)
    ax1.tick_params(axis='y', colors='tab:blue')
    if 'left' in ax1.spines:
        ax1.spines['left'].set_color('tab:blue')
    ax1.yaxis.set_major_locator(ticker.MaxNLocator(nbins=6))

    # Highlight periods near GW peak (±30 days around each flagged center)
    spans_this_ax = 0
    if 'gw_level_near_peak' in df_well.columns:
        for _, row in df_well[df_well['gw_level_near_peak'] == 1].iterrows():
            center_date = pd.to_datetime(row['year_month'])
            h_span = ax1.axvspan(
                center_date - pd.Timedelta(days=30),
                center_date + pd.Timedelta(days=30),
                color='green', alpha=0.2,
                label=('Near GW Peak Window'
                       if spans_this_ax == 0 and not any(lab == 'Near GW Peak Window' for _, lab in legend_pairs)
                       else '')
            )
            spans_this_ax += 1
            if spans_this_ax == 1 and h_span.get_label():
                legend_pairs.append((h_span, 'Near GW Peak Window'))

    # RIGHT axis: Sinkhole counts as bars + risk threshold (red)
    ax2 = ax1.twinx()
    h_bar = ax2.bar(
        df_well['year_month'],
        df_well['sink_count'],
        width=20, alpha=0.8, label='Sinkhole Count', color='tab:red'
    )
    ax2.tick_params(axis='y', colors='tab:red')
    if 'right' in ax2.spines:
        ax2.spines['right'].set_color('tab:red')
    ax2.yaxis.set_major_locator(ticker.MaxNLocator(nbins=6))

    # Threshold line
    h_thr = ax2.axhline(
        threshold, linestyle='--', linewidth=1.5,
        label='Sinkhole Risk Threshold', color='black'
    )

    # --- Grids: horizontal only, driven by the sinkhole (right) axis ---
    ax1.grid(False)  # ensure no grid from left axis
    ax2.grid(axis='y', linestyle='--', alpha=0.6)  # horizontal only
    ax2.set_axisbelow(True)

    # Legend items (only once globally)
    if not any(lab == 'GW Level (m ASL)' for _, lab in legend_pairs):
        legend_pairs.append((h_gw[0], 'GW Level (m ASL)'))
    if not any(lab == 'Sinkhole Count' for _, lab in legend_pairs):
        legend_pairs.append((h_bar, 'Sinkhole Count'))
    if not any(lab == 'Sinkhole Risk Threshold' for _, lab in legend_pairs):
        legend_pairs.append((h_thr, 'Sinkhole Risk Threshold'))

    # Subplot cosmetics
    ax1.set_title(f"Well {well_id}", pad=6)
    ax1.margins(x=0.01)

    # Subplot label (a), (b), ...
    ax1.text(
        0.01, 1.02, f"({chr(97 + idx)})",
        transform=ax1.transAxes,
        fontsize=16, fontweight='bold', va='bottom', ha='left', clip_on=False
    )

    # Only bottom subplot shows x tick labels to save space
    if idx < n_plots - 1:
        ax1.tick_params(axis='x', which='both', labelbottom=False)

# -------------------- SHARED AXIS & LABELS --------------------
# Shared x-axis formatting on the bottom subplot
axes[-1].xaxis.set_major_locator(locator)
axes[-1].xaxis.set_major_formatter(formatter)
axes[-1].set_xlabel("Date")

# ONE figure-wide y-label per side
fig.supylabel("GW Level (m ASL)", x=0.08, fontsize=16, color='tab:blue')
fig.text(0.96, 0.5, "Sinkhole Count", rotation=270,
         va='center', ha='right', fontsize=16, color='tab:red')

# -------------------- LEGEND (two rows) --------------------
unique = OrderedDict()
for h, lab in legend_pairs:
    if lab and lab not in unique:
        unique[lab] = h

# Reserve minimal space at bottom; more room for plots
fig.tight_layout(rect=(0.07, 0.08, 0.93, 1.00))

# Force a two-row legend by choosing columns = ceil(n_items / 2)
n_items = len(unique)
ncol = (n_items + 1) // 2  # integer ceil for 2 rows

fig.legend(
    list(unique.values()), list(unique.keys()),
    loc='lower center',
    ncol=ncol,
    frameon=False,
    columnspacing=1.2,
    handlelength=2.0,
    bbox_to_anchor=(0.5, 0.035)
)

# -------------------- EXPORT --------------------
out_path = "gw_level_peaks_sinkhole.jpeg"
fig.savefig(out_path, bbox_inches='tight', pad_inches=0.3)
plt.show()

## Table 3 — Classification performance by feature set

This table is generated directly from `results['general_summary']`.

In [ ]:
table_3 = results['general_summary'].copy()
display(table_3.round(2))

## Figure 15 — Normalized confusion matrices

The manuscript reports eight feature sets in the order CGS, CS, Climatic, GS, GWS, GWS-Seasonal, Groundwater and TWS.

In [ ]:
per_well_df = results.get("per_well_accuracy", pd.DataFrame()).copy()

# Require the new per-class columns from the updated table
required_cols = [
    "Feature Set", "well_no",
    "Low Risk Count", "High Risk Count",
    "Recall (Low)", "Recall (High)"
]
missing = [c for c in required_cols if c not in per_well_df.columns]
if missing:
    raise ValueError(
        f"The per_well_accuracy table is missing required columns: {missing}. "
        "Make sure you ran the updated evaluate_feature_sets() that adds per-class metrics."
    )

# If there are duplicates (e.g., repeated runs), keep the best-per-well row
# Prefer ROC AUC when available; otherwise Accuracy
per_well_df["_primary_score"] = np.where(
    per_well_df["ROC AUC"].notna(), per_well_df["ROC AUC"], per_well_df["Accuracy"]
)
per_well_best = (
    per_well_df.sort_values(["Feature Set", "well_no", "_primary_score"], ascending=[True, True, False])
               .drop_duplicates(subset=["Feature Set", "well_no"], keep="first")
)

# Aggregate confusion matrices per Feature Set using recalls + true class counts
# Confusion matrix layout:
#               Pred Low   Pred High
# True Low          a           b
# True High         c           d
# With:
#   a = recall_low  * N_low
#   d = recall_high * N_high
#   b = N_low  - a
#   c = N_high - d
confusion_matrices_agg = []  # list of (feature_set, cm_2x2)

for feat_name, g in per_well_best.groupby("Feature Set"):
    cm_sum = np.zeros((2, 2), dtype=float)
    for _, r in g.iterrows():
        N0 = float(r["Low Risk Count"])
        N1 = float(r["High Risk Count"])
        rec0 = float(r["Recall (Low)"])
        rec1 = float(r["Recall (High)"])
        if np.isnan(N0) or np.isnan(N1) or np.isnan(rec0) or np.isnan(rec1):
            continue

        a = max(0.0, min(rec0 * N0, N0))
        d = max(0.0, min(rec1 * N1, N1))
        b = max(0.0, N0 - a)
        c = max(0.0, N1 - d)

        cm_sum += np.array([[a, b], [c, d]], dtype=float)

    confusion_matrices_agg.append((feat_name, cm_sum))

# -------------------- Plot confusion matrices (normalized) --------------------
# Font size controls
SUPTITLE_FS = 16
TITLE_FS = 18
LABEL_FS = 20
TICK_FS = 16
ANNOT_FS = 18
SUBPLOT_LETTER_FS = 18

mpl.rcParams.update({
    "axes.titlesize": TITLE_FS,
    "axes.labelsize": LABEL_FS,
    "xtick.labelsize": TICK_FS,
    "ytick.labelsize": TICK_FS,
    "figure.titlesize": SUPTITLE_FS,
})

n_sets = len(confusion_matrices_agg)
cols = 2
rows = math.ceil(n_sets / cols)
fig, axes = plt.subplots(rows, cols, figsize=(10, 5 * rows), constrained_layout=True)

# Handle the case of a single subplot (axes may not be an array)
axes_flat = np.ravel(axes) if isinstance(axes, np.ndarray) else np.array([axes])

for i, (ax, (name, cm_sum)) in enumerate(zip(axes_flat, confusion_matrices_agg)):
    # Normalize by row (true label)
    with np.errstate(invalid='ignore', divide='ignore'):
        row_sums = cm_sum.sum(axis=1, keepdims=True)
        cm_norm = cm_sum / row_sums
    cm_norm = np.nan_to_num(cm_norm)

    disp = ConfusionMatrixDisplay(cm_norm, display_labels=['Low risk', 'High risk'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format=".2f")

    # --- remove gridlines (and optionally the box) ---
    ax.grid(False, which='both')

    # Remove per-subplot axis labels so we can use shared labels
    ax.set_xlabel('')
    ax.set_ylabel('')

    # Title notes aggregation logic
    ax.set_title(f"{name}", fontsize=TITLE_FS)

    # Tick label size
    ax.tick_params(axis='both', labelsize=TICK_FS)

    # Resize annotation numbers inside cells
    if getattr(disp, "text_", None) is not None:
        for txt in np.ravel(disp.text_):
            txt.set_fontsize(ANNOT_FS)

    # Subplot letter
    subplot_letter = string.ascii_lowercase[i]
    ax.text(-0.14, 0.95, f"({subplot_letter})", transform=ax.transAxes,
            fontsize=SUBPLOT_LETTER_FS, fontweight='bold', va='top', ha='left')

# Remove any unused subplots
total_axes = rows * cols
for j in range(n_sets, total_axes):
    fig.delaxes(axes_flat[j])

# shared x/y labels
fig.supxlabel("Predicted label", fontsize=LABEL_FS)
fig.supylabel("True label", fontsize=LABEL_FS)

plt.savefig("confusion_matrices_normalized__best_params_aggregated.jpeg", dpi=600, bbox_inches='tight')
plt.show()

## Table 4 — Best-performing model by well

The table selects the best feature set per well by the reproduced test ROC AUC logic. `random_state` is omitted from the display because it is a fixed reproducibility setting rather than a tuned hyperparameter.

In [ ]:
table_4 = results['best_config_per_well'].copy()

def _clean_rf_params(value):
    try:
        params = json.loads(value) if isinstance(value, str) else dict(value)
    except Exception:
        return value
    params.pop('random_state', None)
    return json.dumps(params, sort_keys=True)

table_4['best_params'] = table_4['best_params'].apply(_clean_rf_params)
table_4 = table_4.rename(columns={
    'well_no': 'Well No.',
    'feature_set': 'Feature Set',
    'accuracy': 'Accuracy',
    'roc_auc': 'ROC AUC',
    'best_params': 'Best Hyperparameters',
})
display(table_4[['Well No.', 'Feature Set', 'Accuracy', 'ROC AUC', 'Best Hyperparameters']].round(2))

# Part IV — Supplementary and exploratory diagnostics

These analyses are useful for interpretation and quality control but are kept outside the main executable modeling sequence and after the manuscript outputs.

## A. Imputation VIF diagnostics

In [ ]:
def compute_avg_vif_table(vif_all):
    data = {}
    for feature_set_name, vif_dicts in vif_all.items():
        if not vif_dicts:
            continue
        vif_df = pd.DataFrame(vif_dicts)
        data[(feature_set_name, 'mean')] = vif_df.mean().round(1)
        data[(feature_set_name, 'std')] = vif_df.std().round(1)
    return pd.DataFrame(data).sort_index(axis=1, level=0)

avg_vif_df = compute_avg_vif_table(vif_all)
avg_vif_df.to_csv('avg_vif_imput.csv')
avg_vif_df

## B. Risk-model VIF diagnostics

In [ ]:
def calculate_vif(df):
    """
    Calculate Variance Inflation Factor (VIF) for each feature in a DataFrame.
    """
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df)

    vif_data = pd.DataFrame()
    vif_data["feature"] = df.columns
    vif_data["VIF"] = [variance_inflation_factor(X_scaled, i) for i in range(X_scaled.shape[1])]
    return vif_data


def check_vif_for_feature_sets_all_wells(df, feature_sets):
    """
    Calculate VIF for each feature set across all wells, and return the average VIFs.
    """
    all_vif_records = []

    for well, group_df in df.groupby('well_no'):
        numeric_df = group_df.select_dtypes(include=[np.number]).dropna()
        for set_name, features in feature_sets.items():
            try:
                subset = numeric_df[features].dropna()
                if subset.shape[0] < 2:
                    continue  # Skip if not enough rows to compute VIF
                vif_df = calculate_vif(subset)
                vif_df["feature_set"] = set_name
                vif_df["well_no"] = well
                all_vif_records.append(vif_df)
            except KeyError as e:
                print(f"❌ Missing columns in feature set '{set_name}' for well '{well}': {e}")
            except Exception as e:
                print(f"⚠️ Error computing VIF for well '{well}' and feature set '{set_name}': {e}")

    if all_vif_records:
        full_vif_df = pd.concat(all_vif_records, ignore_index=True)

        # Average VIFs across wells
        avg_vif_df = (
            full_vif_df
            .groupby(['feature_set', 'feature'], as_index=False)
            .agg(avg_vif=('VIF', 'mean'), std_vif=('VIF', 'std'), count=('VIF', 'count'))
        )
        return avg_vif_df, full_vif_df
    else:
        return pd.DataFrame(columns=["feature_set", "feature", "avg_vif", "std_vif", "count"]), pd.DataFrame()

# Example usage:
if __name__ == "__main__":
    # Run VIF analysis across all wells
    avg_vif_df, all_vif_df = check_vif_for_feature_sets_all_wells(df_work, feature_sets)

    # Save to CSV
    avg_vif_df.to_csv('vif_for_risk_assesment_features.csv', index=False)
    # all_vif_df.to_csv("all_vif_by_well.csv", index=False)

    # Show final result
    # print("\n📊 Average VIF Across All Wells:")
    print(avg_vif_df)

## C. Extra Trees impurity-based feature importance

In [ ]:
plot_keys = list(feature_importances_all.keys())[:2]

if len(plot_keys) < 2:
    print('Need at least two models to plot one-above-the-other.')
else:
    LABEL_FS, TICK_FS, SUBPLOT_LABEL_FS = 15, 14, 16
    fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharey=True)
    axes = [axes] if not isinstance(axes, (list, np.ndarray)) else axes

    for idx, key in enumerate(plot_keys):
        df_feat = pd.DataFrame(feature_importances_all[key]).fillna(0)
        mean_importance = df_feat.mean().sort_values(ascending=False)
        std_importance = df_feat.std().reindex(mean_importance.index)
        ax = axes[idx]
        positions = range(len(mean_importance))

        ax.bar(positions, mean_importance.values, yerr=std_importance.values, capsize=5)
        ax.set_xticks(positions)
        ax.set_xticklabels(mean_importance.index, rotation=45, ha='right', fontsize=TICK_FS)
        ax.tick_params(axis='y', labelsize=TICK_FS)
        ax.grid(axis='y', linestyle='--', alpha=0.6)
        if idx == 0:
            ax.set_ylabel('Mean Feature Importance', fontsize=LABEL_FS)
        ax.text(
            0.01, 1.02, f'({chr(97 + idx)})', transform=ax.transAxes,
            fontsize=SUBPLOT_LABEL_FS, fontweight='bold', va='bottom', ha='left', clip_on=False,
        )

    plt.tight_layout()
    plt.subplots_adjust(hspace=1.10, bottom=0.18)
    plt.savefig('feature_importance_comparison_imput.jpeg', dpi=600, bbox_inches='tight')
    plt.show()

## D. Correlation of engineered numeric variables with monthly sinkhole count

In [ ]:
# ---- 1. Select numeric features only ----
# Exclude identifiers and non-numeric fields
numeric_df = df_work.select_dtypes(include=[np.number])

# ---- 2. Compute correlations ----
correlation_matrix = numeric_df.corr()

# ---- 3. Extract correlation with target ----
target_correlations = correlation_matrix['sink_count'].drop('sink_count').sort_values(key=abs, ascending=False)

# ---- 4. Plot bar chart ----
plt.figure(figsize=(10, 10))
sns.barplot(x=target_correlations.values, y=target_correlations.index)
plt.title('Feature Correlation with sink_count')
plt.xlabel('Pearson Correlation', fontsize=14)
plt.ylabel('Features', fontsize=14)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)
plt.tight_layout()
plt.grid(True)
plt.show()

## E. Groundwater level versus sinkhole count by well

In [ ]:
# Unique wells and configuration
wells = df_work['well_no'].dropna().unique()
plots_per_fig = 8
cols = 2
rows = math.ceil(plots_per_fig / cols)

# Split into batches of 8
for fig_start in range(0, len(wells), plots_per_fig):
    fig_wells = wells[fig_start:fig_start + plots_per_fig]
    fig, axes = plt.subplots(rows, cols, figsize=(14, rows * 4))
    axes = axes.flatten()

    for i, well in enumerate(fig_wells):
        ax = axes[i]
        sub_df = df_work[df_work['well_no'] == well].dropna(subset=['gw_level_m_asl_imputed', 'sink_count'])

        if len(sub_df) < 2:
            ax.set_visible(False)
            continue

        X = sub_df[['sink_count']].values
        y = sub_df['gw_level_m_asl_imputed'].values

        model = LinearRegression()
        model.fit(X, y)
        y_pred = model.predict(X)

        r2 = r2_score(y, y_pred)
        mae = mean_absolute_error(y, y_pred)

        sns.scatterplot(ax=ax, data=sub_df, x='sink_count', y='gw_level_m_asl_imputed', label='Data')
        ax.plot(sub_df['sink_count'], y_pred, color='red', label='Trendline')

        # Custom legend entries for R² and MAE
        r2_label = f'R² = {r2:.2f}'
        mae_label = f'MAE = {mae:.2f}'
        dummy_r2 = Line2D([0], [0], color='white', label=r2_label)
        dummy_mae = Line2D([0], [0], color='white', label=mae_label)

        handles, labels = ax.get_legend_handles_labels()
        ax.legend(
            [*handles, dummy_r2, dummy_mae],
            [*labels, dummy_r2.get_label(), dummy_mae.get_label()],
            loc='lower right',
            fontsize=13,           # <-- bigger text
            markerscale=1.6,       # <-- bigger scatter dots in legend
            handlelength=2.5,
            frameon=True, fancybox=True
        )

        ax.set_title(f'{well}', fontsize=24)
        ax.set_xlabel('Sinkhole Count', fontsize=20)
        ax.set_ylabel('GW Level (m ASL)', fontsize=20)

    # Hide any unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.savefig("gw_level_vs_peaks_r2_mae.jpeg", dpi=600)
    plt.show()

## F. Groundwater-level hydrographs by regional stage

Supplementary overview adapted from `4_level(1).ipynb`. This uses the reporting-only full source record and does not feed back into either ML stage.

In [ ]:
# Ensure date column is in datetime format
df['date'] = pd.to_datetime(df['date'])

# Create a copy to preserve the original DataFrame
df_grouped = df_article_full.copy()

# Define custom order
stage_order = ['Daugava', 'Dubnik', 'Plavinas']

# Set Seaborn style
sns.set(style="whitegrid")

# Create a vertically stacked FacetGrid by regional_stage
g = sns.FacetGrid(
    df_grouped,
    row="regional_stage",
    row_order=stage_order,
    height=4,
    aspect=3,
    sharex=True,
    sharey=True
)

# Plot lines
g.map_dataframe(sns.lineplot, x="date", y="gw_level_m_asl", hue="well_no")

# Add a separate legend to each axis
for ax in g.axes.flat:
    handles, labels = ax.get_legend_handles_labels()
    # Only add legend if there are handles
    if handles:
        ax.legend(
            handles, labels, title="Well Number",
            loc='upper right', bbox_to_anchor=(1, 1),
            frameon=True, fontsize='small', title_fontsize='medium'
        )

# Customize plot appearance
g.set_axis_labels("Date", "Groundwater Level (m ASL)")
g.set_titles(row_template="{row_name}")
g.fig.subplots_adjust(hspace=0.3)

# Show plot
plt.show()

## G. Groundwater level versus GWS and TWS

Supplementary relationship plots and R² summary adapted from `4_level(1).ipynb`. These are descriptive diagnostics only.

In [ ]:
# STEP 1: Prepare data
df_article_full['date'] = pd.to_datetime(df_article_full['date'])
df_scatter = df_article_full[['well_no', 'regional_stage', 'date', 'gw_level_m_asl',
                 'gws_mm_tavg_gldas', 'tws_mm_tavg_gldas']].dropna()

# ---- Global font sizes for better visibility ----
import matplotlib as mpl
mpl.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 17
})

# STEP 2: Set up plot grid (3 columns)
wells = df_scatter['well_no'].unique()
ncols = 3
nrows = int(np.ceil(len(wells) / ncols))

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(ncols * 6.5, nrows * 5.5),
    sharex=False,
    sharey=False
)
axes = np.array(axes).reshape(-1)  # flatten robustly

r2_records = []

# STEP 3: Plot per well
for idx, well in enumerate(wells):
    ax = axes[idx]
    data = df_scatter[df_scatter['well_no'] == well]
    x_gws = data['gws_mm_tavg_gldas'].values
    x_tws = data['tws_mm_tavg_gldas'].values
    y = data['gw_level_m_asl'].values
    n_measurements = data['date'].nunique()

    # Regressions
    model_gws = LinearRegression().fit(x_gws.reshape(-1, 1), y)
    model_tws = LinearRegression().fit(x_tws.reshape(-1, 1), y)
    y_pred_gws = model_gws.predict(x_gws.reshape(-1, 1))
    y_pred_tws = model_tws.predict(x_tws.reshape(-1, 1))
    r2_gws = r2_score(y, y_pred_gws)
    r2_tws = r2_score(y, y_pred_tws)

    # Store R² and count
    stage = data['regional_stage'].iloc[0]
    r2_records.append({
        'well_no': well,
        'regional_stage': stage,
        'R2_GWS': r2_gws,
        'R2_TWS': r2_tws,
        'n_measurements': n_measurements
    })

    # KDE plots
    sns.kdeplot(x=x_gws, y=y, fill=True, cmap="Blues", thresh=0.05, levels=100, ax=ax, alpha=0.4)
    sns.kdeplot(x=x_tws, y=y, fill=True, cmap="Oranges", thresh=0.05, levels=100, ax=ax, alpha=0.4)

    # Trendlines
    ax.plot(x_gws, y_pred_gws, color='blue', linewidth=2,
            label=f'GWS R²={r2_gws:.2f} | N={n_measurements}')
    ax.plot(x_tws, y_pred_tws, color='orange', linewidth=2,
            label=f'TWS R²={r2_tws:.2f} | N={n_measurements}')

    ax.set_title(f'Well {well}', fontsize=24)
    ax.set_xlabel('Storage (mm)', fontsize=20)
    ax.set_ylabel('GW Level (m abs)', fontsize=20)
    ax.legend()

# STEP 4: Remove unused axes (if any)
for k in range(len(wells), len(axes)):
    fig.delaxes(axes[k])

plt.tight_layout()
plt.savefig("level_vs_gws_tws.jpeg", dpi=600)
plt.show()

In [ ]:
# STEP 6: R² Summary Table
r2_df = pd.DataFrame(r2_records)

# Group by regional_stage and well_no with average R² and total measurement count
r2_summary = r2_df.groupby(['regional_stage', 'well_no']).agg({
    'R2_GWS': 'mean',
    'R2_TWS': 'mean',
    'n_measurements': 'sum'
}).round(2)
r2_summary.to_csv('level_vs_gws_tws_statistics.csv')
r2_summary